# Chronos-2 Forecasting — DIMER `TASK-INFERENCE` tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/chronos-2-forecasting-pipeline/blob/main/tutorials/chronos_2_forecasting_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-amazon%2Fchronos--2-ffcc4d?style=flat)](https://huggingface.co/amazon/chronos-2) [![Upstream](https://img.shields.io/badge/Upstream-amazon--science%2Fchronos--forecasting-181717?style=flat&logo=github&logoColor=white)](https://github.com/amazon-science/chronos-forecasting) [![arXiv](https://img.shields.io/badge/arXiv-2510.15821-b31b1b.svg)](https://arxiv.org/abs/2510.15821)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** zero-shot probabilistic time-series forecasting (univariate, multi-series, multi-target, optional known-future covariates) with the pinned `amazon/chronos-2` checkpoint

**This notebook is standalone.** It carries the repository's package (7 modules under `src/chronos2_pipeline/`, at revision `b8c7ae39de88`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `95a9710e2596287d08352589f42634fa5abdf0a7` (~478 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

Chronos-2 supplies the pretrained zero-shot forecasting model. This repository supplies DIMER configuration and validation, immutable model pinning/digest checks, normalized outputs, chronological evaluation, baselines, and provenance — all of it carried in this notebook. **No training or fine-tuning occurs**, and **no adaptation occurs:** inference is in-context only, the pinned checkpoint is used as published, and no preprocessing is fitted. The default sample is deterministic synthetic teaching data generated in code (the same bytes the repository checks in as `examples/sample-data/chronos_univariate.csv`); its metrics are tutorial/sanity evidence, not a benchmark claim.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; resolve and digest-verify the immutable `amazon/chronos-2` revision; generate the synthetic sample or bring your own CSV; create a leakage-safe chronological holdout and validate the history into an input manifest; run univariate and **multi-target (Mode C)** zero-shot forecasts; interpret median/quantile outputs and the tutorial metrics against naive baselines through an evaluation report; optionally run known-future covariates (Mode D); and export forecasts plus provenance. By the end you can do each of these without the repository being reachable.

**This notebook does not demonstrate:** classification, anomaly detection, imputation, embeddings, training/fine-tuning, calibrated prediction intervals, and production-fitness claims. Out of scope: monthly/quarterly/yearly/business-day, irregular, and gappy calendars — only fixed-width regular frequencies are accepted.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is the default path; CUDA is used automatically when available. The pinned `torch==2.14.0` install is the largest download of the run, followed by the ~478 MB checkpoint.
- **Knowledge:** basic Python and pandas; what a quantile forecast and a chronological holdout are.
- **Data:** the default sample is a deterministic 96-step hourly series generated in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 CSV with unique headers including `series_id`, `timestamp`, and `target`; timestamps parseable, regular, and contiguous per series; targets finite numeric; further numeric columns are covariates. BYOD is read locally in the notebook runtime and is not sent to an external inference service. Do not upload confidential or restricted data (personal or otherwise sensitive data included) to a hosted notebook environment unless you are authorized to do so.
- **External access:** the Hugging Face Hub only, to fetch the pinned `amazon/chronos-2` snapshot (~478 MB in total) at revision `95a9710e2596…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `pandas`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'chronos-forecasting==2.3.1',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pandas==3.0.5',
    'torch==2.14.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'chronos-2-forecasting-pipeline',
    'repository_revision': 'b8c7ae39de88455cb037674f46ea8c5fecb1451c',
    'embedded_module': 'src/chronos2_pipeline/model.py',
    'embedded_modules': ['src/chronos2_pipeline/errors.py', 'src/chronos2_pipeline/provenance.py', 'src/chronos2_pipeline/config.py', 'src/chronos2_pipeline/model.py', 'src/chronos2_pipeline/validation.py', 'src/chronos2_pipeline/inference.py', 'src/chronos2_pipeline/evaluation.py'],
    'module_sha256': '39e18d89ef4eca69627a2565d8b60e6645b2e51577005a0b6432ed249fc8afd4',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, pandas, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pandas': pandas.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/chronos2_pipeline/` @ `b8c7ae39de88`)

The next 7 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/7:** `src/chronos2_pipeline/errors.py`

In [ ]:
"""Exception hierarchy for the pipeline.

Every failure mode the pipeline is contractually required to detect raises a
distinct, catchable type. Nothing here inherits from ``Warning`` and nothing is
downgraded to a log line: the RFC's position is that a silently-degraded
forecast is worse than no forecast.
"""

from __future__ import annotations

from typing import Any

__all__ = [
    "Chronos2PipelineError",
    "ModelSourceError",
    "ModelIntegrityError",
    "HubUnavailableError",
    "ValidationError",
    "UpstreamContractError",
]


class Chronos2PipelineError(Exception):
    """Base class for every error raised by this package."""


class ModelSourceError(Chronos2PipelineError, ValueError):
    """A model source other than the single pinned revision was requested.

    Also inherits :class:`ValueError` so that callers written against the RFC's
    "``ValueError`` subclass" wording keep working.
    """


class ModelIntegrityError(Chronos2PipelineError, ValueError):
    """The downloaded snapshot does not match the pinned revision or digests."""


class HubUnavailableError(ModelIntegrityError):
    """The Hugging Face Hub could not be reached, so it said nothing at all.

    Deliberately distinct from a Hub that *answered* with something other than
    the pin: an unreachable Hub is an availability failure, a disagreeing Hub is
    a supply-chain failure. :func:`chronos2_pipeline.model.load_pinned_model`
    tolerates the first when every recorded digest still matches — recording in
    provenance that the revision was not confirmed against the Hub on that load
    — and never tolerates the second.

    It subclasses :class:`ModelIntegrityError` so a caller that only wants "the
    model could not be loaded safely" keeps working unchanged, and so
    ``require_hub_confirmation=True`` raises the type callers already catch.
    """


class ValidationError(Chronos2PipelineError, ValueError):
    """A DIMER-side validation rule rejected the request.

    Parameters
    ----------
    code
        Stable machine-readable identifier for the rule that failed, e.g.
        ``"IRREGULAR_FREQUENCY"``. Callers and tests match on this, never on the
        prose message.
    message
        Human-readable explanation.
    details
        Structured context — offending ids, observed values, limits.
    """

    def __init__(self, code: str, message: str, details: dict[str, Any] | None = None) -> None:
        self.code = code
        self.message = message
        self.details: dict[str, Any] = dict(details or {})
        super().__init__(f"[{code}] {message}")

    def __repr__(self) -> str:  # pragma: no cover - debugging aid
        return (
            f"ValidationError(code={self.code!r}, message={self.message!r}, "
            f"details={self.details!r})"
        )


class UpstreamContractError(Chronos2PipelineError, RuntimeError):
    """The pinned upstream release behaved differently from the recorded contract.

    Raised, for example, when ``predict_df`` stops returning the median in its
    ``predictions`` column, or drops a column the normalisation map depends on.
    This is deliberately fatal: it is the tripwire that must fail CI when a
    future upstream version changes semantics underneath us.
    """

**Module 2/7:** `src/chronos2_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
"""Export metadata: what model, what runtime, what request.

A forecast without this block is not reproducible. Everything recorded here is
read from the live objects — resolved library versions, the device the weights
actually landed on, the context and horizon the model actually used — never from
a constant or from what the caller asked for.
"""

from __future__ import annotations

import platform
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _pkg_version
from typing import TYPE_CHECKING, Any

if TYPE_CHECKING:  # pragma: no cover
    pass  # standalone rewrite (build_notebook.py): `from .model import ModelIdentity` removed — names are kernel globals defined by the carried modules

__all__ = ["PROVENANCE_SCHEMA_VERSION", "TRACKED_PACKAGES", "runtime_versions", "build_provenance"]

#: Bump when the shape of the exported dict changes in a way consumers must notice.
#: 1.1 added ``past_covariate_names`` and ``known_future_covariate_names`` when
#: Phase 2 made covariates reachable; a 1.0 consumer reading a covariate-informed
#: forecast cannot tell the two kinds apart, so the addition is one to notice.
PROVENANCE_SCHEMA_VERSION = "1.1"

#: RFC "Runtime/dependency reproducibility": these versions travel with results.
TRACKED_PACKAGES: tuple[str, ...] = (
    "chronos-forecasting",
    "torch",
    "transformers",
    "huggingface-hub",
    "numpy",
    "pandas",
    "accelerate",
    "einops",
    "safetensors",
)


def runtime_versions(packages: tuple[str, ...] = TRACKED_PACKAGES) -> dict[str, str | None]:
    """Resolved distribution versions, ``None`` for anything not installed."""
    resolved: dict[str, str | None] = {"python": platform.python_version()}
    for name in packages:
        try:
            resolved[name] = _pkg_version(name)
        except PackageNotFoundError:
            resolved[name] = None
    return resolved


def build_provenance(
    identity: ModelIdentity,
    *,
    device: str,
    dtype: str,
    n_ids: int,
    n_targets: int,
    n_covariates: int,
    past_covariate_names: list[str],
    known_future_covariate_names: list[str],
    requested_context_length: int | None,
    effective_context_length: int,
    longest_series_length: int,
    shortest_series_length: int,
    requested_prediction_length: int,
    effective_prediction_length: int,
    autoregressive_unrolled: bool,
    requested_quantiles: list[float],
    effective_quantiles: list[float],
    batch_size: int,
    cross_learning: bool,
    latency_seconds: float,
    warm_up_performed: bool,
    frequency: str,
    observed_frequency: str,
) -> dict[str, Any]:
    """Assemble the three-block export metadata dict.

    Notes
    -----
    ``effective_quantiles`` is the list actually present in the output. In v1 it
    always equals ``requested_quantiles``, because out-of-grid levels are refused
    rather than clamped — but both are exported so a future release that permits
    clamping cannot do it invisibly.

    ``effective_context_length`` is the context bound actually in force: the
    request, the model's limit and the longest series in the request, whichever
    binds first. It is not a promise that every series contributed that much —
    a series shorter than it contributed its whole history and no more, which is
    what ``shortest_series_length`` is for.

    ``past_covariate_names`` and ``known_future_covariate_names`` partition the
    request's covariates. RFC Mode D asks that the two be distinguished, and the
    distinction is not recoverable from the forecast frame: neither kind appears
    in the output, so an export recording only ``n_covariates`` cannot say
    whether a covariate was read up to the forecast origin or all the way across
    the horizon — which is the difference between a forecast that could have
    been made in advance and one that could not.

    ``latency_seconds`` times the scored ``predict_df`` call only.
    ``warm_up_performed`` says whether a discarded warm-up call preceded it; a
    cold first call includes lazy CUDA/kernel initialisation and is not a
    comparable latency figure.
    """
    return {
        "schema_version": PROVENANCE_SCHEMA_VERSION,
        "model": identity.as_dict(),
        "runtime": {
            **runtime_versions(),
            "platform": platform.platform(),
            "device": device,
            "dtype": dtype,
        },
        "inference": {
            "n_ids": n_ids,
            "n_targets": n_targets,
            "n_covariates": n_covariates,
            "past_covariate_names": list(past_covariate_names),
            "known_future_covariate_names": list(known_future_covariate_names),
            "requested_context_length": requested_context_length,
            "effective_context_length": effective_context_length,
            "model_context_length": identity.model_context_length,
            "longest_series_length": longest_series_length,
            "shortest_series_length": shortest_series_length,
            "requested_prediction_length": requested_prediction_length,
            "effective_prediction_length": effective_prediction_length,
            "model_prediction_length": identity.model_prediction_length,
            "autoregressive_unrolled": autoregressive_unrolled,
            "requested_quantile_levels": list(requested_quantiles),
            "effective_quantile_levels": list(effective_quantiles),
            "batch_size": batch_size,
            "cross_learning": cross_learning,
            "declared_frequency": frequency,
            "observed_frequency": observed_frequency,
            "latency_seconds": latency_seconds,
            "warm_up_performed": warm_up_performed,
        },
    }

**Module 3/7:** `src/chronos2_pipeline/config.py` (carried verbatim; see the note above)

In [ ]:
"""User-facing forecast configuration.

Mirrors the RFC's "User parameters" block one field at a time. Validation here
covers only the scalar fields — anything that needs the data or the model
(quantile-grid membership, frequency, resource limits) belongs to
:mod:`chronos2_pipeline.validation`, which runs later and has both.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Literal

# standalone rewrite (build_notebook.py): `from .errors import ValidationError` removed — names are kernel globals defined by the carried modules

__all__ = ["ForecastConfig", "DEFAULT_QUANTILE_LEVELS", "SUPPORTED_DEVICES"]

#: RFC default. Kept as a module constant so tests and docs cite one source.
DEFAULT_QUANTILE_LEVELS: tuple[float, ...] = (0.1, 0.5, 0.9)

SUPPORTED_DEVICES: tuple[str, ...] = ("auto", "cpu", "cuda")

DeviceName = Literal["auto", "cpu", "cuda"]


@dataclass(frozen=True)
class ForecastConfig:
    """Configuration for one forecast request.

    Field defaults are exactly the RFC "User parameters" YAML block.

    Notes
    -----
    ``cross_learning`` stays ``False`` in v1 and is not a beginner knob. When it
    is ever flipped, both its value and ``batch_size`` must be exported, because
    cross-learning makes results depend on batch composition.
    """

    id_column: str = "series_id"
    timestamp_column: str = "timestamp"
    target: str | list[str] = "target"
    prediction_length: int = 24
    quantile_levels: list[float] = field(default_factory=lambda: list(DEFAULT_QUANTILE_LEVELS))
    batch_size: int = 256
    context_length: int | None = None
    frequency: str | None = None
    device: DeviceName = "auto"
    cross_learning: bool = False

    def __post_init__(self) -> None:
        self._validate_columns()
        self._validate_prediction_length()
        self._validate_quantiles()
        self._validate_batch_size()
        self._validate_context_length()
        self._validate_device()

    # -- accessors ---------------------------------------------------------

    @property
    def target_names(self) -> list[str]:
        """``target`` normalised to a list, preserving order."""
        if isinstance(self.target, str):
            return [self.target]
        return list(self.target)

    @property
    def n_targets(self) -> int:
        return len(self.target_names)

    @property
    def reserved_columns(self) -> list[str]:
        """Columns that are structural rather than covariates."""
        return [self.id_column, self.timestamp_column, *self.target_names]

    # -- scalar validation -------------------------------------------------

    def _validate_columns(self) -> None:
        names = {
            "id_column": self.id_column,
            "timestamp_column": self.timestamp_column,
        }
        for label, value in names.items():
            if not isinstance(value, str) or not value:
                raise ValidationError(
                    "CONFIG_INVALID_COLUMN",
                    f"{label} must be a non-empty string, got {value!r}.",
                    {"field": label, "value": value},
                )

        targets = self.target
        if isinstance(targets, str):
            targets = [targets]
        if not isinstance(targets, list) or not targets:
            raise ValidationError(
                "CONFIG_INVALID_TARGET",
                "target must be a non-empty string or a non-empty list of strings.",
                {"target": self.target},
            )
        if any(not isinstance(t, str) or not t for t in targets):
            raise ValidationError(
                "CONFIG_INVALID_TARGET",
                "every target name must be a non-empty string.",
                {"target": self.target},
            )
        if len(set(targets)) != len(targets):
            raise ValidationError(
                "CONFIG_INVALID_TARGET",
                "target names must be unique.",
                {"target": self.target},
            )
        overlap = set(targets) & {self.id_column, self.timestamp_column}
        if overlap:
            raise ValidationError(
                "CONFIG_INVALID_TARGET",
                f"target names collide with the id/timestamp columns: {sorted(overlap)}.",
                {"collisions": sorted(overlap)},
            )

    def _validate_prediction_length(self) -> None:
        value = self.prediction_length
        if isinstance(value, bool) or not isinstance(value, int):
            raise ValidationError(
                "CONFIG_INVALID_PREDICTION_LENGTH",
                f"prediction_length must be an int, got {type(value).__name__}.",
                {"prediction_length": value},
            )
        if value <= 0:
            raise ValidationError(
                "CONFIG_INVALID_PREDICTION_LENGTH",
                f"prediction_length must be > 0, got {value}.",
                {"prediction_length": value},
            )

    def _validate_quantiles(self) -> None:
        levels = self.quantile_levels
        if not isinstance(levels, list) or not levels:
            raise ValidationError(
                "CONFIG_INVALID_QUANTILES",
                "quantile_levels must be a non-empty list of floats.",
                {"quantile_levels": levels},
            )
        if any(isinstance(q, bool) or not isinstance(q, (int, float)) for q in levels):
            raise ValidationError(
                "CONFIG_INVALID_QUANTILES",
                "quantile_levels must contain only numbers.",
                {"quantile_levels": levels},
            )
        if len(set(levels)) != len(levels):
            raise ValidationError(
                "CONFIG_INVALID_QUANTILES",
                f"quantile_levels must be unique, got {levels}.",
                {"quantile_levels": levels},
            )
        if list(levels) != sorted(levels):
            raise ValidationError(
                "CONFIG_INVALID_QUANTILES",
                f"quantile_levels must be sorted ascending, got {levels}.",
                {"quantile_levels": levels},
            )
        out_of_range = [q for q in levels if not (0.0 < float(q) < 1.0)]
        if out_of_range:
            raise ValidationError(
                "CONFIG_INVALID_QUANTILES",
                f"quantile_levels must lie strictly inside (0, 1); offenders: {out_of_range}.",
                {"out_of_range": out_of_range},
            )

    def _validate_batch_size(self) -> None:
        value = self.batch_size
        if isinstance(value, bool) or not isinstance(value, int):
            raise ValidationError(
                "CONFIG_INVALID_BATCH_SIZE",
                f"batch_size must be an int, got {type(value).__name__}.",
                {"batch_size": value},
            )
        if value < 1:
            raise ValidationError(
                "CONFIG_INVALID_BATCH_SIZE",
                f"batch_size must be >= 1, got {value}.",
                {"batch_size": value},
            )

    def _validate_context_length(self) -> None:
        value = self.context_length
        if value is None:
            return
        if isinstance(value, bool) or not isinstance(value, int):
            raise ValidationError(
                "CONFIG_INVALID_CONTEXT_LENGTH",
                f"context_length must be an int or None, got {type(value).__name__}.",
                {"context_length": value},
            )
        if value <= 0:
            raise ValidationError(
                "CONFIG_INVALID_CONTEXT_LENGTH",
                f"context_length must be > 0 when set, got {value}.",
                {"context_length": value},
            )

    def _validate_device(self) -> None:
        if self.device not in SUPPORTED_DEVICES:
            raise ValidationError(
                "CONFIG_INVALID_DEVICE",
                f"device must be one of {list(SUPPORTED_DEVICES)}, got {self.device!r}.",
                {"device": self.device},
            )
        if not isinstance(self.cross_learning, bool):
            raise ValidationError(
                "CONFIG_INVALID_CROSS_LEARNING",
                f"cross_learning must be a bool, got {type(self.cross_learning).__name__}.",
                {"cross_learning": self.cross_learning},
            )

**Module 4/7:** `src/chronos2_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
"""Pinned, integrity-verified Chronos-2 loader.

One model, one revision, checked four ways before any weight is deserialised:

1. the requested source must be exactly ``amazon/chronos-2`` at the pinned commit;
2. the directory Hugging Face resolved must be that commit's snapshot;
3. ``model.safetensors`` must match the recorded SHA-256 and byte size;
4. ``config.json`` must match the recorded SHA-256.

Pickle-based weights (``.bin``/``.pt``/``.pth``/``.ckpt``) in the snapshot are a
hard refusal, not a fallback.

The revision SHA is the primary integrity anchor (RFC C-7): it covers repository
configuration as well as the weight file. The digests are secondary, file-level
assertions that additionally catch a corrupted or truncated download.

Check 2 asks the Hub which commit the pinned name resolves to, which means it
needs the network. Availability and integrity are kept apart there (review round
2): a Hub that *answers* with a different commit is a supply-chain failure and
always fails the load, while a Hub that cannot be *reached* at all is an
availability failure — the load proceeds on the digests, which prove the
snapshot's content byte for byte on their own, and the exported provenance
records ``revision_confirmed_against_hub: false`` with the reason. Pass
``require_hub_confirmation=True`` to refuse that degraded mode; it is not the
default, so a cached, digest-matching snapshot still loads offline.
"""

from __future__ import annotations

import hashlib
import json
import os
import re
from collections.abc import Callable
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .errors import HubUnavailableError, ModelIntegrityError, ModelSourceError` removed — names are kernel globals defined by the carried modules

__all__ = [
    "MODEL_ID",
    "MODEL_REVISION",
    "MODEL_LICENSE",
    "MODEL_KEY",
    "PINNED_MODEL_ID",
    "PINNED_REVISION",
    "PINNED_WEIGHTS_SHA256",
    "PINNED_WEIGHTS_BYTES",
    "PINNED_CONFIG_SHA256",
    "PINNED_LICENSE",
    "PINNED_LICENSE_SOURCE",
    "PINNED_MODEL_URL",
    "PINNED_MODEL_KEY",
    "DEFAULT_WEIGHTS_DIR",
    "MANIFEST_NAME",
    "EXPECTED_TRAINED_QUANTILES",
    "WEIGHTS_FILENAME",
    "CONFIG_FILENAME",
    "REFUSED_WEIGHT_SUFFIXES",
    "ModelIdentity",
    "LoadedModel",
    "sha256_file",
    "check_model_source",
    "verify_snapshot",
    "stage_missing_files",
    "resolve_hub_revision",
    "load_pinned_model",
]

# --------------------------------------------------------------------------
# Supply-chain pins. Independently verified against the Hugging Face Hub on
# 2026-09-08; see MODEL_CARD.md for the provenance record.
# --------------------------------------------------------------------------

#: Fleet identity names (DIMER NOTEBOOK_SPEC 1.1 ST3): the four constants every DIMER package
#: spells the same way, so a generated standalone notebook and the fleet tooling can read the
#: identity without knowing this package's own vocabulary. The ``PINNED_*`` names below are the
#: package's original spelling and stay in use everywhere; they alias these, never the reverse.
MODEL_ID = "amazon/chronos-2"
MODEL_REVISION = "95a9710e2596287d08352589f42634fa5abdf0a7"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "chronos-2"

PINNED_MODEL_ID = MODEL_ID
PINNED_REVISION = MODEL_REVISION
PINNED_WEIGHTS_SHA256 = "ddcda3c7508bf2528087723e98a20707cc04b7f370ae275a9fd88078ddba4f42"
PINNED_WEIGHTS_BYTES = 477_930_472
PINNED_CONFIG_SHA256 = "ef1143bfdc9c0376d9a056eefca46cb4b1ec3d0ffacd541ff56feb40fb708031"

PINNED_LICENSE = MODEL_LICENSE
#: There is no LICENSE file in the Hugging Face repository at the pinned
#: revision — the files present are .gitattributes, README.md, config.json and
#: model.safetensors. The licence is declared in the model card metadata only,
#: and this string is what gets exported rather than a claim about a file.
PINNED_LICENSE_SOURCE = (
    f"apache-2.0 declared in the Hugging Face model card metadata of "
    f"{PINNED_MODEL_ID} at revision {PINNED_REVISION}; the repository contains "
    f"no LICENSE file at that revision"
)
PINNED_MODEL_URL = f"https://huggingface.co/{PINNED_MODEL_ID}/tree/{PINNED_REVISION}"

#: The 21-level grid Chronos-2 was trained on, read from the pinned config.json.
#: Held here only so a test can assert the loaded pipeline still reports it; the
#: runtime value always comes from the model, never from this constant.
EXPECTED_TRAINED_QUANTILES: tuple[float, ...] = (
    0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5,
    0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99,
)  # fmt: skip

WEIGHTS_FILENAME = "model.safetensors"
CONFIG_FILENAME = "config.json"

#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files also live in a
#: repository-local snapshot directory named by this key, described by a committed
#: ``dimer-base-manifest.json``. The manifest is the parity anchor a standalone notebook
#: carries inline; the digest constants above are asserted equal to it on every load, so the
#: two can never disagree silently.
PINNED_MODEL_KEY = MODEL_KEY
MANIFEST_NAME = "dimer-base-manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative

#: Pickle-format checkpoints execute arbitrary code on load. If any appear in
#: the snapshot the load is refused rather than silently preferring safetensors.
REFUSED_WEIGHT_SUFFIXES: tuple[str, ...] = (".bin", ".pt", ".pth", ".ckpt", ".pkl")

_MUTABLE_REFS = frozenset({"main", "master", "latest", "head", "HEAD"})
_URI_SCHEME = re.compile(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://")
_WINDOWS_DRIVE = re.compile(r"^[a-zA-Z]:[\\/]")
_SHA1_HEX = re.compile(r"^[0-9a-f]{40}$")

#: Exception type names that mean the request never got an answer out of the
#: Hub: no DNS, no route, no socket, a proxy or TLS failure, or a cache that has
#: been told not to go to the network at all (``HF_HUB_OFFLINE=1``).
_UNREACHABLE_EXC_NAMES = frozenset(
    {
        "OfflineModeIsEnabled",
        "LocalEntryNotFoundError",
        "ConnectionError",
        "ConnectTimeout",
        "ReadTimeout",
        "ConnectTimeoutError",
        "Timeout",
        "TimeoutError",
        "ProxyError",
        "SSLError",
        "ChunkedEncodingError",
        "socket.timeout",
    }
)

#: HTTP statuses that are the Hub failing to serve rather than the Hub
#: disagreeing. 401/403/404 are excluded on purpose: they are answers — the
#: repo is gated, or the revision is not there — and must fail closed.
_UNREACHABLE_HTTP_STATUS = frozenset({408, 425, 429, 500, 502, 503, 504})


def _hub_unreachable_reason(exc: BaseException) -> str | None:
    """Say why the Hub was unreachable, or ``None`` if it actually answered.

    The distinction is the whole point of review round 2. ``None`` means the Hub
    (or its cache of an authoritative answer) responded and the response was not
    the pin — gated repo, missing repo, missing revision, malformed reply — which
    is a supply-chain event and must never be downgraded to "offline".
    """
    response = getattr(exc, "response", None)
    status = getattr(response, "status_code", None)
    if status is None:
        status = getattr(exc, "status_code", None)
    if isinstance(status, int):
        # An HTTP status means something answered, so only the serving failures
        # count as unreachable.
        if status in _UNREACHABLE_HTTP_STATUS:
            return f"the Hub responded HTTP {status} rather than serving the lookup"
        return None

    names = {cls.__name__ for cls in type(exc).__mro__}
    if names & _UNREACHABLE_EXC_NAMES:
        return f"{type(exc).__name__}: {exc}"
    # requests' transport errors all descend from OSError; anything left that is
    # an OSError is a socket/filesystem failure, not a Hub answer.
    if isinstance(exc, OSError):
        return f"{type(exc).__name__}: {exc}"
    return None


@dataclass(frozen=True)
class ModelIdentity:
    """Everything a downstream export needs to say which model produced a result."""

    name: str
    revision: str
    config_sha256: str
    weights_sha256: str
    weights_bytes: int
    license: str
    license_source: str
    source_url: str
    trained_quantiles: tuple[float, ...]
    model_context_length: int
    model_prediction_length: int
    #: True only when a Hub lookup ran on *this* load and answered with the
    #: pinned commit. False whenever the Hub was not reached, so the exported
    #: metadata can never imply a check that did not happen. The default is the
    #: honest one: an identity built by hand has confirmed nothing.
    revision_confirmed_against_hub: bool = False
    #: How the revision was established on this load — the successful lookup, or
    #: the reason it could not run and what carried the integrity claim instead.
    revision_confirmation_note: str = "no Hub confirmation was attempted"
    snapshot_path: str = field(default="", compare=False)

    @property
    def model_id(self) -> str:
        """Alias for name for compatibility with downstream loaders."""
        return self.name

    def as_dict(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "revision": self.revision,
            "config_sha256": self.config_sha256,
            "weights_sha256": self.weights_sha256,
            "weights_bytes": self.weights_bytes,
            "license": self.license,
            "license_source": self.license_source,
            "source_url": self.source_url,
            "revision_confirmed_against_hub": self.revision_confirmed_against_hub,
            "revision_confirmation_note": self.revision_confirmation_note,
            "trained_quantiles": list(self.trained_quantiles),
            "model_context_length": self.model_context_length,
            "model_prediction_length": self.model_prediction_length,
        }


@dataclass(frozen=True)
class LoadedModel:
    """A loaded pipeline plus the identity that was proven before loading it."""

    pipeline: Any
    identity: ModelIdentity
    device: str
    dtype: str

    @property
    def trained_quantiles(self) -> tuple[float, ...]:
        return self.identity.trained_quantiles

    @property
    def model_context_length(self) -> int:
        return self.identity.model_context_length

    @property
    def model_prediction_length(self) -> int:
        return self.identity.model_prediction_length


def sha256_file(path: str | os.PathLike[str], *, chunk_size: int = 1 << 20) -> str:
    """SHA-256 of a file's bytes, streamed so a 478 MB weight file is cheap."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def _looks_like_local_path(model_id: str) -> bool:
    if _WINDOWS_DRIVE.match(model_id):
        return True
    if "\\" in model_id:
        return True
    if model_id.startswith(("/", "~", "./", "../", ".\\", "..\\")):
        return True
    if model_id in (".", ".."):
        return True
    # A bare "a/b" repo id that also happens to exist on disk is still a path.
    try:
        return Path(model_id).exists()
    except OSError:  # pragma: no cover - pathological names only
        return False


def check_model_source(model_id: Any, revision: Any) -> None:
    """Refuse anything but the single pinned ``(model_id, revision)`` pair.

    Raises
    ------
    ModelSourceError
        For URI schemes (``s3://``, ``hf://``, ``http://``, ``file://`` ...),
        local filesystem paths, any repo id other than ``amazon/chronos-2``,
        mutable refs (``main``, ``latest``), and any other revision.
    """
    if not isinstance(model_id, str) or not model_id:
        raise ModelSourceError(
            f"model_id must be the string {PINNED_MODEL_ID!r}; got {model_id!r}."
        )

    scheme = _URI_SCHEME.match(model_id)
    if scheme:
        raise ModelSourceError(
            f"Refusing model source {model_id!r}: URI schemes such as "
            f"{scheme.group(0)!r} are not permitted in the standard path. Only "
            f"the pinned Hugging Face repo {PINNED_MODEL_ID!r} at revision "
            f"{PINNED_REVISION} is accepted."
        )

    if _looks_like_local_path(model_id):
        raise ModelSourceError(
            f"Refusing model source {model_id!r}: local filesystem paths are not "
            f"permitted in the standard path, because they carry no revision to "
            f"verify. Only {PINNED_MODEL_ID!r} at revision {PINNED_REVISION} is "
            f"accepted."
        )

    if model_id != PINNED_MODEL_ID:
        raise ModelSourceError(
            f"Refusing model source {model_id!r}: this pipeline is pinned to "
            f"{PINNED_MODEL_ID!r} and does not load other checkpoints."
        )

    if revision is None or not isinstance(revision, str) or not revision:
        raise ModelSourceError(
            f"A revision is required and must be the pinned commit "
            f"{PINNED_REVISION}; got {revision!r}."
        )

    if revision in _MUTABLE_REFS or not _SHA1_HEX.match(revision):
        raise ModelSourceError(
            f"Refusing mutable or non-commit revision {revision!r}: the pinned "
            f"commit SHA is the primary supply-chain invariant. Use "
            f"{PINNED_REVISION}."
        )

    if revision != PINNED_REVISION:
        raise ModelSourceError(
            f"Refusing revision {revision!r}: this pipeline is pinned to "
            f"{PINNED_REVISION}. A different revision must go through an RFC "
            f"update, not a keyword argument."
        )


def resolve_hub_revision(model_id: str, revision: str) -> str:
    """Ask the Hub which commit ``revision`` names, independently of the download.

    This is the oracle the revision check needs. ``snapshot_download`` lays a
    snapshot out under ``snapshots/<requested-sha>/``, so when the request *is* a
    SHA the directory name is that SHA by construction and comparing the two
    proves nothing (review R-5). ``model_info(...).sha`` is the commit the Hub
    itself says it served, so a hub answering the pinned name with a different
    commit is detectable.

    Raises
    ------
    HubUnavailableError
        The Hub was never reached — offline mode, no route, a proxy or TLS
        failure, a rate limit, a 5xx. Nothing is claimed about the commit; the
        caller decides whether the recorded digests are enough (they are, by
        default: see :func:`load_pinned_model`).
    ModelIntegrityError
        The Hub answered and its answer was not usable as a confirmation — a
        gated or missing repo, a revision it does not have, or a reply carrying
        no commit SHA. This always fails closed.
    """
    from huggingface_hub import HfApi

    try:
        info = HfApi().model_info(repo_id=model_id, revision=revision)
    except Exception as exc:  # noqa: BLE001 - classified, then re-raised
        unreachable = _hub_unreachable_reason(exc)
        if unreachable is not None:
            raise HubUnavailableError(
                f"Could not reach the Hugging Face Hub to confirm {model_id!r}@{revision}: "
                f"{unreachable}. No claim is made about the resolved commit."
            ) from exc
        raise ModelIntegrityError(
            f"The Hub refused to resolve {model_id!r}@{revision}: "
            f"{type(exc).__name__}: {exc}. The Hub answered and its answer was not the "
            f"pinned commit, so the load is refused."
        ) from exc
    sha = getattr(info, "sha", None)
    if not isinstance(sha, str) or not sha:
        raise ModelIntegrityError(
            f"The Hub returned no commit SHA for {model_id!r}@{revision}; the resolved "
            f"revision cannot be verified against the pin."
        )
    return sha


def _read_manifest(root: Path, *, expected_revision: str) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    try:
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
    except (OSError, ValueError) as exc:
        raise ModelIntegrityError(
            f"Could not read snapshot manifest {manifest_path}: {exc}"
        ) from exc
    if manifest.get("modelId") != PINNED_MODEL_ID:
        raise ModelIntegrityError(
            f"Snapshot manifest names model {manifest.get('modelId')!r}, not the pinned "
            f"{PINNED_MODEL_ID!r} ({manifest_path})."
        )
    if manifest.get("revision") != expected_revision:
        raise ModelIntegrityError(
            f"Snapshot manifest names revision {manifest.get('revision')!r}, not the pinned "
            f"{expected_revision!r} ({manifest_path})."
        )
    if not isinstance(manifest.get("files"), list) or not manifest["files"]:
        raise ModelIntegrityError(f"Snapshot manifest lists no files: {manifest_path}")
    return manifest


def _verify_manifest_snapshot(
    root: Path,
    *,
    expected_revision: str,
    expected_config_sha256: str,
    expected_weights_sha256: str,
    expected_weights_bytes: int,
    resolved_revision: str | None,
) -> dict[str, Any]:
    """Manifest-driven verification of a snapshot directory named by ``PINNED_MODEL_KEY``.

    The directory name carries no revision here — the committed manifest does — so the
    revision assertion moves to the manifest and the per-file digests come from it. The
    package's own ``PINNED_*`` digest constants are then asserted **equal to** the manifest
    entries, so this path is never weaker than the revision-directory path.
    """
    if resolved_revision is not None and resolved_revision != expected_revision:
        raise ModelIntegrityError(
            f"The Hub resolved this model to revision {resolved_revision!r}, which does not "
            f"match the pinned revision {expected_revision!r}. The pinned commit is the primary "
            f"supply-chain invariant, so a different commit is refused whatever its contents."
        )
    manifest = _read_manifest(root, expected_revision=expected_revision)

    refused = sorted(
        p.name
        for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in REFUSED_WEIGHT_SUFFIXES
    )
    if refused:
        raise ModelIntegrityError(
            f"Refusing to load: pickle-format weight files are present in the snapshot and "
            f"would permit arbitrary code execution: {refused}. Only {WEIGHTS_FILENAME} is "
            f"acceptable."
        )

    digests: dict[str, str] = {}
    for entry in manifest["files"]:
        path = root / entry["path"]
        if not path.is_file():
            raise ModelIntegrityError(f"Snapshot file listed in the manifest is missing: {path}")
        actual_bytes = path.stat().st_size
        if actual_bytes != entry["bytes"]:
            raise ModelIntegrityError(
                f"{entry['path']} is {actual_bytes} bytes, expected {entry['bytes']} for "
                f"revision {expected_revision}."
            )
        digest = sha256_file(path)
        if digest != entry["sha256"]:
            raise ModelIntegrityError(
                f"{entry['path']} SHA-256 {digest} does not match the manifest digest "
                f"{entry['sha256']}."
            )
        digests[entry["path"]] = digest

    sizes = {entry["path"]: int(entry["bytes"]) for entry in manifest["files"]}
    for filename in (CONFIG_FILENAME, WEIGHTS_FILENAME):
        if filename not in digests:
            raise ModelIntegrityError(
                f"Snapshot manifest does not list {filename}, which the loader requires "
                f"({root / MANIFEST_NAME})."
            )
    if digests[CONFIG_FILENAME] != expected_config_sha256:
        raise ModelIntegrityError(
            f"Manifest {CONFIG_FILENAME} digest {digests[CONFIG_FILENAME]} does not match the "
            f"pinned digest {expected_config_sha256}."
        )
    if digests[WEIGHTS_FILENAME] != expected_weights_sha256:
        raise ModelIntegrityError(
            f"Manifest {WEIGHTS_FILENAME} digest {digests[WEIGHTS_FILENAME]} does not match the "
            f"pinned digest {expected_weights_sha256}."
        )
    if sizes[WEIGHTS_FILENAME] != expected_weights_bytes:
        raise ModelIntegrityError(
            f"Manifest {WEIGHTS_FILENAME} size {sizes[WEIGHTS_FILENAME]} does not match the "
            f"pinned byte count {expected_weights_bytes}."
        )

    return {
        "path": str(root),
        "revision": str(manifest["revision"]),
        "config_sha256": digests[CONFIG_FILENAME],
        "weights_sha256": digests[WEIGHTS_FILENAME],
        "weights_bytes": sizes[WEIGHTS_FILENAME],
        "files": list(manifest["files"]),
        "model_key": manifest.get("modelKey", PINNED_MODEL_KEY),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at ``PINNED_REVISION`` straight into ``root``."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(PINNED_MODEL_ID, relative_path, revision=PINNED_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | os.PathLike[str] | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent from the snapshot directory.

    A fresh clone commits the manifest and git-ignores the weights, so this is how the
    snapshot is populated. Only files named by the manifest are fetched, only at
    ``PINNED_REVISION``, and :func:`verify_snapshot` still re-hashes everything afterwards.
    Returns the relative paths fetched (empty when nothing was missing).
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root, expected_revision=PINNED_REVISION)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them "
            f"at {PINNED_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def verify_snapshot(
    snapshot_path: str | os.PathLike[str],
    *,
    expected_revision: str = PINNED_REVISION,
    expected_config_sha256: str = PINNED_CONFIG_SHA256,
    expected_weights_sha256: str = PINNED_WEIGHTS_SHA256,
    expected_weights_bytes: int = PINNED_WEIGHTS_BYTES,
    check_resolved_revision: bool = True,
    resolved_revision: str | None = None,
) -> dict[str, Any]:
    """Prove a downloaded snapshot directory is the pinned revision, byte for byte.

    Parameters
    ----------
    resolved_revision
        The commit an *independent* resolution says was served — normally
        :func:`resolve_hub_revision`'s answer. When given, it is compared against
        ``expected_revision`` and the snapshot directory name must agree with it
        too. When omitted, the directory name is used alone, which is the weaker
        check: ``huggingface_hub`` names the directory after the *requested*
        revision, so with a SHA request that comparison is a tautology and cannot
        fail (review R-5). Production goes through :func:`load_pinned_model`,
        which always supplies it.

    Returns
    -------
    dict
        ``{"revision", "config_sha256", "weights_sha256", "weights_bytes"}``.
    """
    root = Path(snapshot_path)
    if not root.is_dir():
        raise ModelIntegrityError(f"Model snapshot path is not a directory: {root}")

    if (root / MANIFEST_NAME).is_file():
        # Fleet snapshot directory (``weights/<PINNED_MODEL_KEY>/``): the revision is carried by
        # the committed manifest rather than by the directory name, and the per-file digests come
        # from it. The pinned constants are asserted equal to the manifest inside.
        return _verify_manifest_snapshot(
            root,
            expected_revision=expected_revision,
            expected_config_sha256=expected_config_sha256,
            expected_weights_sha256=expected_weights_sha256,
            expected_weights_bytes=expected_weights_bytes,
            resolved_revision=resolved_revision if check_resolved_revision else None,
        )

    if check_resolved_revision:
        if resolved_revision is not None and resolved_revision != expected_revision:
            raise ModelIntegrityError(
                f"The Hub resolved this model to revision {resolved_revision!r}, which "
                f"does not match the pinned revision {expected_revision!r}. The pinned "
                f"commit is the primary supply-chain invariant, so a different commit is "
                f"refused whatever its contents."
            )
        if root.name != expected_revision:
            raise ModelIntegrityError(
                f"Resolved model revision {root.name!r} does not match the "
                f"pinned revision {expected_revision!r} (snapshot at {root})."
            )
    resolved_revision = resolved_revision or root.name

    refused = sorted(
        p.name
        for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in REFUSED_WEIGHT_SUFFIXES
    )
    if refused:
        raise ModelIntegrityError(
            f"Refusing to load: pickle-format weight files are present in the "
            f"snapshot and would permit arbitrary code execution: {refused}. "
            f"Only {WEIGHTS_FILENAME} is acceptable."
        )

    weights = root / WEIGHTS_FILENAME
    if not weights.is_file():
        raise ModelIntegrityError(f"Missing {WEIGHTS_FILENAME} in snapshot {root}.")

    actual_bytes = weights.stat().st_size
    if actual_bytes != expected_weights_bytes:
        raise ModelIntegrityError(
            f"{WEIGHTS_FILENAME} is {actual_bytes} bytes, expected "
            f"{expected_weights_bytes} for revision {expected_revision}."
        )

    weights_sha256 = sha256_file(weights)
    if weights_sha256 != expected_weights_sha256:
        raise ModelIntegrityError(
            f"{WEIGHTS_FILENAME} SHA-256 {weights_sha256} does not match the "
            f"pinned digest {expected_weights_sha256}."
        )

    config = root / CONFIG_FILENAME
    if not config.is_file():
        raise ModelIntegrityError(f"Missing {CONFIG_FILENAME} in snapshot {root}.")

    config_sha256 = sha256_file(config)
    if config_sha256 != expected_config_sha256:
        raise ModelIntegrityError(
            f"{CONFIG_FILENAME} SHA-256 {config_sha256} does not match the "
            f"pinned digest {expected_config_sha256}."
        )

    return {
        "revision": resolved_revision,
        "config_sha256": config_sha256,
        "weights_sha256": weights_sha256,
        "weights_bytes": actual_bytes,
    }


def resolve_device(device: str) -> str:
    """Turn ``"auto"`` into a concrete device name without importing torch eagerly."""
    if device != "auto":
        return device
    import torch

    return "cuda" if torch.cuda.is_available() else "cpu"


def _read_trained_quantiles(pipeline: Any) -> tuple[float, ...]:
    quantiles = getattr(pipeline, "quantiles", None)
    if quantiles is None:  # pragma: no cover - upstream contract change
        raise ModelIntegrityError(
            "Loaded pipeline does not expose a `quantiles` attribute; the "
            "trained quantile grid cannot be read from the pinned model."
        )
    return tuple(float(q) for q in quantiles)


def _describe_dtype(pipeline: Any) -> tuple[str, str]:
    """Return ``(device, dtype)`` actually in effect, read off the loaded weights."""
    model = getattr(pipeline, "model", None) or getattr(pipeline, "inner_model", None)
    if model is None:  # pragma: no cover - upstream contract change
        return ("unknown", "unknown")
    try:
        param = next(model.parameters())
    except (StopIteration, AttributeError):  # pragma: no cover
        return ("unknown", "unknown")
    return (str(param.device), str(param.dtype))


def load_pinned_model(
    model_id: str = PINNED_MODEL_ID,
    revision: str = PINNED_REVISION,
    *,
    device: str = "auto",
    dtype: str = "auto",
    cache_dir: str | os.PathLike[str] | None = None,
    revision_resolver: Callable[[str, str], str] = resolve_hub_revision,
    require_hub_confirmation: bool = False,
    weights_dir: str | os.PathLike[str] | None = None,
    allow_download: bool = False,
) -> LoadedModel:
    """Download, verify and load the pinned Chronos-2 checkpoint.

    Parameters
    ----------
    model_id, revision
        Present so that a caller *can* pass something else — and be refused with
        a clear message. Both default to the pin.
    device
        ``"auto"``, ``"cpu"`` or ``"cuda"``. Recorded as resolved, not as asked.
    dtype
        Passed to ``BaseChronosPipeline.from_pretrained``; ``"auto"`` follows the
        checkpoint. The dtype actually in effect is read back off the weights.
    revision_resolver
        How the resolved commit is established, independently of the download.
        Defaults to :func:`resolve_hub_revision`, which asks the Hub. Injectable
        so the mismatch path is testable without a hostile hub; production has no
        reason to pass anything else.
    require_hub_confirmation
        When ``True``, a Hub that cannot be reached is fatal. The default
        ``False`` is the available mode: an unreachable Hub lets the load
        continue on the recorded digests alone, and the resulting
        :class:`ModelIdentity` reports ``revision_confirmed_against_hub=False``
        with the reason. A Hub that *answers* with a different commit is refused
        in either mode.
    weights_dir
        A fleet snapshot directory holding ``dimer-base-manifest.json`` (normally
        ``weights/<PINNED_MODEL_KEY>/``). When given, nothing is resolved through
        ``snapshot_download``: missing manifest entries are staged with
        :func:`stage_missing_files`, :func:`verify_snapshot` re-hashes every entry against the
        manifest *and* against the pinned digest constants, and the same
        ``BaseChronosPipeline.from_pretrained`` call loads from that directory. Identical files,
        identical loader, a different place to keep them.
    allow_download
        Only meaningful with ``weights_dir``: permit :func:`stage_missing_files` to fetch the
        manifest entries that are absent, at ``PINNED_REVISION``. The default refuses.

    Raises
    ------
    ModelSourceError
        The source is not the pinned pair.
    HubUnavailableError
        ``require_hub_confirmation=True`` and the Hub could not be reached.
    ModelIntegrityError
        The Hub resolved the pin to another commit, or a file digest or the
        weight byte count does not match the pin.
    """
    check_model_source(model_id, revision)

    from chronos import BaseChronosPipeline

    hub_revision: str | None
    snapshot_path: str
    if weights_dir is not None:
        # Fleet snapshot path: the committed manifest is the revision oracle and the digests
        # prove the content, so no Hub lookup is performed for the identity.
        root = Path(weights_dir)
        stage_missing_files(root, allow_download=allow_download)
        verified = verify_snapshot(root)
        hub_revision = None
        confirmation_note = (
            f"the Hub was not consulted on this load; the snapshot's identity rests on the "
            f"committed {MANIFEST_NAME} at {root}, whose per-file SHA-256 digests were "
            f"re-hashed and asserted equal to the pinned {CONFIG_FILENAME} and "
            f"{WEIGHTS_FILENAME} digests and the weight byte count"
        )
        snapshot_path = str(root)
    else:
        from huggingface_hub import snapshot_download

        try:
            hub_revision = revision_resolver(model_id, revision)
        except HubUnavailableError as exc:
            if require_hub_confirmation:
                raise
            hub_revision = None
            confirmation_note = (
                f"the Hub was not consulted successfully on this load ({exc}); the "
                f"snapshot's identity rests on the recorded {CONFIG_FILENAME} and "
                f"{WEIGHTS_FILENAME} SHA-256 digests and the weight byte count, which "
                f"prove its content independently of the Hub"
            )
        else:
            if hub_revision != PINNED_REVISION:
                raise ModelIntegrityError(
                    f"The Hub resolved {model_id!r}@{revision} to commit {hub_revision!r}, not "
                    f"the pinned {PINNED_REVISION!r}. Nothing is downloaded and nothing is loaded."
                )
            confirmation_note = (
                f"the Hub resolved {model_id}@{revision} to {hub_revision} on this load"
            )

        snapshot_path = snapshot_download(
            repo_id=model_id,
            revision=revision,
            cache_dir=cache_dir,
        )
        verified = verify_snapshot(snapshot_path, resolved_revision=hub_revision)

    resolved_device = resolve_device(device)
    pipeline = BaseChronosPipeline.from_pretrained(
        str(snapshot_path),
        device_map=resolved_device,
        dtype=dtype,
    )

    actual_device, actual_dtype = _describe_dtype(pipeline)
    identity = ModelIdentity(
        name=model_id,
        revision=verified["revision"],
        config_sha256=verified["config_sha256"],
        weights_sha256=verified["weights_sha256"],
        weights_bytes=verified["weights_bytes"],
        license=PINNED_LICENSE,
        license_source=PINNED_LICENSE_SOURCE,
        source_url=PINNED_MODEL_URL,
        revision_confirmed_against_hub=hub_revision is not None,
        revision_confirmation_note=confirmation_note,
        trained_quantiles=_read_trained_quantiles(pipeline),
        model_context_length=int(pipeline.model_context_length),
        model_prediction_length=int(pipeline.model_prediction_length),
        snapshot_path=str(snapshot_path),
    )
    return LoadedModel(
        pipeline=pipeline,
        identity=identity,
        device=actual_device,
        dtype=actual_dtype,
    )

**Module 5/7:** `src/chronos2_pipeline/validation.py` (carried verbatim; see the note above)

In [ ]:
"""DIMER-side validation — RFC rules 1-21.

Every check here runs *before* ``predict_df`` and is independent of the pinned
upstream validator. Two reasons the duplication is deliberate:

* Upstream's ``freq=`` argument bypasses frequency inference entirely. Its own
  docstring says so (``chronos/chronos2/pipeline.py`` L881-885 in 2.3.1: "the
  provided ``freq`` is used as-is and is not checked against the data, even when
  ``validate_inputs=True``"). Supplying ``frequency`` must not be a way to smuggle
  a gappy series past a regularity check, so rules 7-10 are ours (RFC C-3).
* Upstream raises bare ``ValueError`` with prose messages. DIMER needs stable
  machine-readable codes for its own error surface.

Frequency is established by explicit diff equality — ``diff(timestamps)`` must
have exactly one distinct value per series — not by ``pd.infer_freq``, which
tolerates patterns this contract rejects and needs three points before it will
say anything at all.

**Phase 1 supports fixed-width frequencies only.** A frequency is fixed-width
when one period is always the same ``Timedelta``: "15min", "h", "D", "7D". Data
spaced monthly, quarterly, yearly or business-daily has no constant period, so
diff equality can never hold for it; it is rejected with
``CALENDAR_FREQUENCY_UNSUPPORTED`` rather than mislabelled "irregular".
Widening the rule to calendar offsets is a Phase-2 design decision; see
``MODEL_CARD.md`` and ``README.md``, which state the limitation. Weekly data is
in scope — a week is a constant seven days — even though the pandas *alias*
``W`` is anchored and therefore not fixed-width; declare it as ``"7D"``.

The same rule governs a *declared* ``frequency``: an alias that is not
fixed-width cannot be compared against the observed ``Timedelta``, and upstream
would use it as-is to build the horizon index, so it is rejected
(``FREQUENCY_NOT_FIXED_WIDTH``) rather than forwarded unchecked. Only a declared
alias that has been affirmatively confirmed equal to the observed interval is
passed to ``predict_df`` (see :attr:`ValidationResult.confirmed_frequency`).
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any

import numpy as np
import pandas as pd
import pandas.api.types as ptypes

# standalone rewrite (build_notebook.py): `from .config import ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import ValidationError` removed — names are kernel globals defined by the carried modules

__all__ = [
    "ResourceLimits",
    "ValidationResult",
    "DEFAULT_LIMITS",
    "MIN_OBSERVATIONS",
    "validate_forecast_request",
    "quantiles_in_grid",
    "nearest_grid_level",
    "INPUT_SCHEMA",
    "validate_inputs",
]

#: RFC rule 10. Upstream's own frequency inference needs three points
#: (``chronos/df_utils.py`` L30), and a two-point series has no repeated
#: interval to validate, so regularity is unfalsifiable below three.
MIN_OBSERVATIONS = 3

#: How close a rejected level has to be to a grid member before the error
#: message names that member as the probable intent. Purely cosmetic: it never
#: widens what is accepted (see :func:`quantiles_in_grid`).
_QUANTILE_HINT_TOL = 1e-6


@dataclass(frozen=True)
class ResourceLimits:
    """DIMER-side resource guards (RFC rule 20).

    Deliberately conservative: these bound what a single request may ask of a
    shared serving process, and are separate from the model's own limits.
    """

    max_ids: int = 1000
    max_targets: int = 64
    max_covariates: int = 64
    max_rows: int = 5_000_000
    max_context_length: int = 8192
    #: Deliberately **above** the model's native horizon (1024) so that the
    #: unroll gate below, not this guard, is what answers a request beyond the
    #: model's capacity. With the two equal, ``allow_unroll`` was unreachable
    #: under the default limits and ``autoregressive_unrolled`` could never be
    #: ``True`` on the shipped configuration (review R-4).
    max_prediction_length: int = 4096
    min_observations: int = MIN_OBSERVATIONS


DEFAULT_LIMITS = ResourceLimits()


@dataclass(frozen=True)
class ValidationResult:
    """The normalised request, plus everything derived while proving it valid."""

    history: pd.DataFrame
    future: pd.DataFrame | None
    frequency: pd.Timedelta
    series_ids: list[Any]
    target_names: list[str]
    covariate_names: list[str]
    n_rows: int
    max_series_length: int
    min_series_length: int
    requested_quantiles: list[float] = field(default_factory=list)
    #: Covariates whose future values the caller supplied, i.e. the covariate
    #: columns present in the future table. Upstream decides this the same way:
    #: ``[c for c in covariate_columns if c in future_df.columns]``
    #: (``chronos/chronos2/preprocess.py`` L195).
    known_future_covariate_names: list[str] = field(default_factory=list)
    #: Covariates the model may read up to the forecast origin and no further.
    #: Every covariate is one or the other, so these two partition
    #: :attr:`covariate_names` (RFC Mode D asks that they be distinguished).
    past_covariate_names: list[str] = field(default_factory=list)
    #: The caller's declared ``frequency`` alias, and **only** when it was
    #: affirmatively confirmed equal to ``frequency``. ``None`` whenever the
    #: caller declared nothing. Nothing else may be forwarded to ``predict_df``,
    #: whose own docstring says ``freq`` "is used as-is and is not checked
    #: against the data, even when ``validate_inputs=True``" (review R-1).
    confirmed_frequency: str | None = None

    @property
    def n_ids(self) -> int:
        return len(self.series_ids)

    @property
    def n_targets(self) -> int:
        return len(self.target_names)

    @property
    def n_covariates(self) -> int:
        return len(self.covariate_names)

    @property
    def n_known_future_covariates(self) -> int:
        return len(self.known_future_covariate_names)

    @property
    def n_past_covariates(self) -> int:
        return len(self.past_covariate_names)


def quantiles_in_grid(
    requested: list[float], grid: tuple[float, ...] | list[float]
) -> list[float]:
    """Return the requested levels that are *not* in the trained grid.

    Membership is **exact float identity**, matching the gate upstream itself
    uses (``set(quantile_levels).issubset(training_quantile_levels)`` at
    ``chronos/chronos2/pipeline.py`` L797 in 2.3.1). A tolerance here would
    accept levels that upstream then routes to ``interpolate_quantiles``
    silently — e.g. ``0.1 * 7 == 0.7000000000000001`` is not ``0.7`` — which is
    exactly the silent substitution RFC C-5 exists to prevent (review R-7).
    """
    grid_set = {float(g) for g in grid}
    return [float(q) for q in requested if float(q) not in grid_set]


def nearest_grid_level(level: float, grid: tuple[float, ...] | list[float]) -> float | None:
    """The grid member a rejected level was probably meant to be, if any is close."""
    if len(grid) == 0:
        return None
    grid_array = np.asarray(grid, dtype=float)
    nearest = float(grid_array[int(np.argmin(np.abs(grid_array - float(level))))])
    if abs(nearest - float(level)) <= _QUANTILE_HINT_TOL:
        return nearest
    return None


def _fail(code: str, message: str, **details: Any) -> None:
    raise ValidationError(code, message, details)


# --------------------------------------------------------------------------
# Rules 1-6: shape, parsing, duplicates, ordering
# --------------------------------------------------------------------------


def _require_columns(df: pd.DataFrame, required: list[str], code: str, table: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        _fail(
            code,
            f"{table} table is missing required column(s): {missing}.",
            table=table,
            missing=missing,
            present=list(df.columns),
        )


def _parse_timestamps(df: pd.DataFrame, column: str, table: str) -> pd.Series:
    try:
        parsed = pd.to_datetime(df[column], errors="raise")
    except (ValueError, TypeError, pd.errors.ParserError) as exc:
        _fail(
            "TIMESTAMP_UNPARSEABLE",
            f"{table} column {column!r} could not be parsed as timestamps: {exc}",
            table=table,
            column=column,
        )
    if isinstance(parsed, pd.DataFrame):  # duplicate column names
        _fail(
            "TIMESTAMP_UNPARSEABLE",
            f"{table} column {column!r} is duplicated; a single timestamp column is required.",
            table=table,
            column=column,
        )
    if parsed.isna().any():
        n_bad = int(parsed.isna().sum())
        _fail(
            "TIMESTAMP_UNPARSEABLE",
            f"{table} column {column!r} has {n_bad} unparseable/null timestamp(s).",
            table=table,
            column=column,
            n_null=n_bad,
        )
    return parsed


def _coerce_targets(df: pd.DataFrame, targets: list[str], policy: str) -> pd.DataFrame:
    """RFC rule 4 and rule 19: explicit numeric and missing-value policies.

    ``policy="strict"`` (the only v1 policy) rejects any value that is not
    numeric-coercible, and rejects nulls outright — v1 does not impute
    (RFC "explicitly out of scope": silent interpolation).
    """
    if policy != "strict":
        _fail(
            "TARGET_POLICY_UNSUPPORTED",
            f"target coercion policy {policy!r} is not supported in v1; only 'strict' is.",
            policy=policy,
        )
    out = df.copy()
    for col in targets:
        original = out[col]
        coerced = pd.to_numeric(original, errors="coerce")
        newly_bad = coerced.isna() & original.notna()
        if newly_bad.any():
            offenders = original[newly_bad].unique()[:5].tolist()
            _fail(
                "TARGET_NOT_NUMERIC",
                f"target column {col!r} contains {int(newly_bad.sum())} value(s) that are "
                f"not numeric-coercible under the 'strict' policy; e.g. {offenders}.",
                column=col,
                n_offending=int(newly_bad.sum()),
                examples=[str(v) for v in offenders],
            )
        if coerced.isna().any():
            _fail(
                "TARGET_MISSING_VALUES",
                f"target column {col!r} contains {int(coerced.isna().sum())} missing value(s). "
                f"v1 rejects gaps and nulls rather than interpolating them.",
                column=col,
                n_missing=int(coerced.isna().sum()),
            )
        if not np.isfinite(coerced.to_numpy(dtype=float)).all():
            _fail(
                "TARGET_NOT_FINITE",
                f"target column {col!r} contains non-finite values (inf/-inf).",
                column=col,
            )
        out[col] = coerced.astype(float)
    return out


def _coerce_covariates(
    df: pd.DataFrame, covariates: list[str], policy: str, table: str
) -> pd.DataFrame:
    """Rules 4 and 19 applied to covariate columns (Phase 2).

    **Phase 2 admits numeric covariates only.** Upstream would accept a string
    or categorical covariate, but it encodes one by a route that depends on how
    many targets the request has: ``target_encode = use_target_encoding and
    n_targets == 1`` (``chronos/chronos2/preprocess.py`` L415), so the same
    holiday-name column is target-encoded in a single-target request and
    ordinal-encoded in a two-target one. A covariate whose meaning changes with
    an unrelated field of the request is not a contract this phase can export
    honestly, so non-numeric covariates are refused by name
    (``COVARIATE_NOT_NUMERIC``) rather than silently encoded. "Numeric" is
    judged the way :func:`_coerce_targets` judges it — by whether coercion loses
    a value, not by dtype — so a column of numeric strings from a CSV is
    accepted and a column of category labels is not. Temporal columns are named
    separately, before that coercion can turn them into nanoseconds.

    ``bool`` is deliberately treated as numeric and coerced to ``0.0``/``1.0``.
    Upstream classes it as *categorical* (``is_numeric_dtype(c) and not
    is_bool_dtype(c)`` at preprocess.py L188), so an unconverted boolean flag
    would take exactly the target-count-dependent path above. Coercing it here
    pins it to the numeric path in both cases, which is also the reading the
    RFC's own covariate example uses (``holiday,0``).

    Nulls and non-finite values are refused for the same reason they are in
    targets: v1 does not impute, and upstream would carry a NaN covariate into
    the forecast without complaint.
    """
    if policy != "strict":
        _fail(
            "TARGET_POLICY_UNSUPPORTED",
            f"covariate coercion policy {policy!r} is not supported in v1; only 'strict' is.",
            policy=policy,
        )
    reason = (
        "upstream would encode a categorical covariate by a route that differs between a "
        "single-target and a multi-target request, so its exported meaning would depend on "
        "how many targets were asked for"
    )
    out = df.copy()
    for col in covariates:
        original = out[col]
        if ptypes.is_bool_dtype(original):
            coerced = original.astype(float)
        elif ptypes.is_numeric_dtype(original):
            coerced = pd.to_numeric(original, errors="coerce")
        elif ptypes.is_datetime64_any_dtype(original) or ptypes.is_timedelta64_dtype(original):
            #: Named before the coercion below, which would turn a temporal column into
            #: nanoseconds-since-epoch and accept it as an ordinary numeric covariate.
            _fail(
                "COVARIATE_NOT_NUMERIC",
                f"{table} covariate column {col!r} has dtype {original.dtype}. A temporal "
                f"column is not a numeric covariate; coercing it would feed the model "
                f"nanoseconds since the epoch. Derive the feature you mean (hour of day, "
                f"days to event) as a numeric column instead.",
                table=table,
                column=col,
                dtype=str(original.dtype),
            )
        else:
            #: Same policy as targets: a column of numeric strings is coerced, a column
            #: that is genuinely categorical is refused. Judged by whether coercion loses
            #: a value, not by dtype, so a CSV read as ``object`` is not refused for it.
            coerced = pd.to_numeric(original, errors="coerce")
            newly_bad = coerced.isna() & original.notna()
            if newly_bad.any():
                offenders = original[newly_bad].unique()[:5].tolist()
                _fail(
                    "COVARIATE_NOT_NUMERIC",
                    f"{table} covariate column {col!r} contains "
                    f"{int(newly_bad.sum())} value(s) that are not numeric-coercible; e.g. "
                    f"{offenders}. Phase 2 supports numeric covariates only: {reason}. "
                    f"Encode it yourself (e.g. 0/1 indicator columns) to pin the "
                    f"representation.",
                    table=table,
                    column=col,
                    dtype=str(original.dtype),
                    n_offending=int(newly_bad.sum()),
                    examples=[str(v) for v in offenders],
                )
        if coerced.isna().any():
            _fail(
                "COVARIATE_MISSING_VALUES",
                f"{table} covariate column {col!r} contains "
                f"{int(coerced.isna().sum())} missing value(s). v1 rejects gaps and nulls "
                f"rather than interpolating them; upstream would carry the NaN into the "
                f"forecast without complaint.",
                table=table,
                column=col,
                n_missing=int(coerced.isna().sum()),
            )
        if not np.isfinite(coerced.to_numpy(dtype=float)).all():
            _fail(
                "COVARIATE_NOT_FINITE",
                f"{table} covariate column {col!r} contains non-finite values (inf/-inf).",
                table=table,
                column=col,
            )
        out[col] = coerced.astype(float)
    return out


# --------------------------------------------------------------------------
# Rules 7-10: regularity, gaps, shared frequency, minimum length
# --------------------------------------------------------------------------


def _fixed_width(offset: Any) -> pd.Timedelta | None:
    """One period of ``offset`` as a ``Timedelta``, or ``None`` if it has no fixed width.

    ``to_offset("h").nanos`` is 3.6e12; ``to_offset("ME").nanos`` raises, because
    a month is not a constant duration. That distinction is the whole Phase-1
    frequency contract.
    """
    try:
        return pd.Timedelta(offset.nanos, unit="ns")
    except (ValueError, AttributeError):
        return None


def _calendar_alias(timestamps: pd.Series) -> str | None:
    """The pandas alias of a series that is calendar-regular but not fixed-width.

    Used only to give monthly/quarterly/business-daily data an honest error
    instead of calling it "irregular" (review R-3). Returns ``None`` for
    anything ``pd.infer_freq`` cannot name, and for anything it names that *is*
    fixed-width (which diff equality would already have accepted).
    """
    try:
        alias = pd.infer_freq(pd.DatetimeIndex(timestamps))
    except (ValueError, TypeError):
        return None
    if alias is None:
        return None
    try:
        offset = pd.tseries.frequencies.to_offset(alias)
    except (ValueError, TypeError):  # pragma: no cover - infer_freq returns parseable aliases
        return None
    return None if _fixed_width(offset) is not None else alias


def _series_frequency(timestamps: pd.Series, series_id: Any, min_observations: int) -> pd.Timedelta:
    """One series' frequency, by explicit diff equality. Rules 7, 9 and 10."""
    n = len(timestamps)
    if n < min_observations:
        _fail(
            "SERIES_TOO_SHORT",
            f"series {series_id!r} has {n} observation(s); at least {min_observations} are "
            f"required before a frequency can be validated.",
            series_id=str(series_id),
            n_observations=n,
            minimum=min_observations,
        )

    diffs = timestamps.diff().dropna()
    distinct = pd.Series(diffs.unique())
    if len(distinct) != 1:
        deltas = sorted(pd.Timedelta(d) for d in distinct)
        calendar = _calendar_alias(timestamps)
        if calendar is not None:
            _fail(
                "CALENDAR_FREQUENCY_UNSUPPORTED",
                f"series {series_id!r} is regular on the calendar frequency {calendar!r}, "
                f"which has no fixed period ({[str(d) for d in deltas]}). Phase 1 supports "
                f"fixed-width frequencies only (e.g. '15min', 'h', 'D', '7D'); calendar "
                f"frequencies such as monthly, quarterly, yearly and business-daily are "
                f"out of scope for v1 and are documented as such in MODEL_CARD.md.",
                series_id=str(series_id),
                inferred_alias=calendar,
                observed_intervals=[str(d) for d in deltas],
            )
        base = deltas[0]
        is_gapped = base > pd.Timedelta(0) and all(
            (d % base) == pd.Timedelta(0) for d in deltas
        )
        if is_gapped:
            _fail(
                "SERIES_GAP",
                f"series {series_id!r} has missing periods: observed intervals "
                f"{[str(d) for d in deltas]} are all multiples of {base}, so timestamps "
                f"are skipped. v1 rejects gaps rather than interpolating them.",
                series_id=str(series_id),
                base_interval=str(base),
                observed_intervals=[str(d) for d in deltas],
            )
        _fail(
            "IRREGULAR_FREQUENCY",
            f"series {series_id!r} is irregular: observed intervals "
            f"{[str(d) for d in deltas]} are not all equal.",
            series_id=str(series_id),
            observed_intervals=[str(d) for d in deltas],
        )

    freq = pd.Timedelta(distinct.iloc[0])
    if freq <= pd.Timedelta(0):
        _fail(
            "IRREGULAR_FREQUENCY",
            f"series {series_id!r} has a non-positive interval {freq}.",
            series_id=str(series_id),
            observed_intervals=[str(freq)],
        )
    return freq


def _shared_frequency(
    df: pd.DataFrame, config: ForecastConfig, limits: ResourceLimits
) -> tuple[pd.Timedelta, int, int]:
    """Per-series frequency (rules 7, 9, 10) plus cross-series agreement (rule 8)."""
    frequencies: dict[Any, pd.Timedelta] = {}
    lengths: list[int] = []
    for series_id, block in df.groupby(config.id_column, sort=True, observed=True):
        stamps = block[config.timestamp_column]
        lengths.append(len(stamps))
        frequencies[series_id] = _series_frequency(stamps, series_id, limits.min_observations)

    distinct = sorted({str(f) for f in frequencies.values()})
    if len(distinct) > 1:
        _fail(
            "MIXED_FREQUENCY",
            f"all series in one request must share a frequency; observed {distinct}.",
            frequencies={str(k): str(v) for k, v in frequencies.items()},
            distinct=distinct,
        )
    return next(iter(frequencies.values())), max(lengths), min(lengths)


def _check_declared_frequency(declared: str | None, observed: pd.Timedelta) -> str | None:
    """A declared ``frequency`` must agree with the data — it may not override it.

    Returns the alias only when it has been affirmatively confirmed equal to
    ``observed``; the caller forwards nothing else to ``predict_df``.

    Every path either confirms or fails. A non-fixed calendar alias ("W", "ME",
    "QS", "B", "YE") used to fall through this function untouched and was then
    handed to ``predict_df``, which builds the horizon index from it without
    checking it against the data — an hourly series declared ``frequency="ME"``
    came back stamped at month ends, with no error (review R-1). Since the
    observed frequency is always a fixed ``Timedelta`` by construction (see
    :func:`_series_frequency`), a non-fixed alias can never agree with it, so
    the answer is always rejection.
    """
    if declared is None:
        return None
    try:
        offset = pd.tseries.frequencies.to_offset(declared)
    except (ValueError, TypeError) as exc:
        _fail(
            "FREQUENCY_UNPARSEABLE",
            f"frequency {declared!r} is not a valid pandas offset alias: {exc}",
            frequency=declared,
        )
    declared_delta = _fixed_width(offset)
    if declared_delta is None:
        _fail(
            "FREQUENCY_NOT_FIXED_WIDTH",
            f"declared frequency {declared!r} is a calendar offset with no fixed period, "
            f"so it cannot agree with the interval observed in the data ({observed}). "
            f"Upstream would use it as-is to lay out the forecast horizon without "
            f"checking it against the data, moving the forecast onto a different time "
            f"axis. Phase 1 supports fixed-width frequencies only, so declare the "
            f"interval the data actually has ({observed}) as a fixed-width alias "
            f"instead (RFC C-3).",
            declared=declared,
            observed_interval=str(observed),
        )
    if declared_delta != observed:
        _fail(
            "FREQUENCY_MISMATCH",
            f"declared frequency {declared!r} ({declared_delta}) does not match the "
            f"interval observed in the data ({observed}). Supplying `frequency` does "
            f"not override the data (RFC C-3).",
            declared=declared,
            declared_interval=str(declared_delta),
            observed_interval=str(observed),
        )
    return declared


# --------------------------------------------------------------------------
# Rules 15-18: the future-covariate table
# --------------------------------------------------------------------------


def _validate_future(
    future_df: pd.DataFrame,
    history: pd.DataFrame,
    config: ForecastConfig,
    frequency: pd.Timedelta,
    series_ids: list[Any],
    covariates: list[str],
    target_policy: str,
) -> tuple[pd.DataFrame, list[str]]:
    _require_columns(
        future_df,
        [config.id_column, config.timestamp_column],
        "FUTURE_MISSING_COLUMNS",
        "future",
    )

    leaked = [c for c in config.target_names if c in future_df.columns]
    if leaked:
        _fail(
            "FUTURE_TARGET_LEAKAGE",
            f"future table contains target column(s) {leaked}. Future target values are "
            f"forbidden as inputs.",
            leaked_columns=leaked,
        )

    extra = [c for c in future_df.columns if c not in history.columns]
    if extra:
        _fail(
            "FUTURE_COLUMN_NOT_IN_HISTORY",
            f"future covariate column(s) {extra} do not exist in the historical table. "
            f"The pinned upstream validator requires future columns to be a subset of "
            f"historical columns (RFC C-4).",
            extra_columns=extra,
            historical_columns=list(history.columns),
        )

    #: Which covariates the future table actually carries — upstream's own rule
    #: (``preprocess.py`` L195). A future table that carries none is a no-op
    #: upstream: every covariate stays past-only and the table changes nothing
    #: but the cost of validating it. Supplying one is a statement that some
    #: covariate is known ahead, so an empty one is refused rather than
    #: silently ignored.
    known_future = [c for c in covariates if c in future_df.columns]
    if not known_future:
        _fail(
            "FUTURE_TABLE_HAS_NO_COVARIATES",
            f"future table carries no covariate column: it has "
            f"{sorted(set(future_df.columns) - {config.id_column, config.timestamp_column})!r} "
            f"beyond the id and timestamp columns, and the historical covariates are "
            f"{covariates!r}. Upstream would treat every covariate as past-only and the "
            f"table would change nothing, so it is refused rather than silently ignored.",
            future_columns=list(future_df.columns),
            historical_covariates=covariates,
        )

    future = future_df.copy()
    future[config.timestamp_column] = _parse_timestamps(future, config.timestamp_column, "future")
    future = _coerce_covariates(future, known_future, target_policy, "future")

    if future[config.id_column].isna().any():
        _fail(
            "NULL_IDS",
            "future table contains null series identifiers.",
            table="future",
        )

    #: Compared as values, not as ``str``. Stringifying made historical id ``1``
    #: and future id ``"1"`` equal, which they are not: upstream joins the two
    #: tables on the raw values and would find no match (review R-13).
    future_ids = set(future[config.id_column].unique())
    history_ids = set(series_ids)
    if future_ids != history_ids:
        future_only = sorted(str(v) for v in future_ids - history_ids)
        history_only = sorted(str(v) for v in history_ids - future_ids)
        _fail(
            "FUTURE_ID_MISMATCH",
            f"future table ids must equal historical ids exactly, compared by value and "
            f"type. Only in future: {future_only[:5]}; only in history: {history_only[:5]}.",
            future_only=future_only[:20],
            history_only=history_only[:20],
        )

    future = future.sort_values(
        [config.id_column, config.timestamp_column], kind="mergesort"
    ).reset_index(drop=True)

    horizon = config.prediction_length
    last_seen = history.groupby(config.id_column, observed=True)[config.timestamp_column].max()

    for series_id, block in future.groupby(config.id_column, sort=True, observed=True):
        if len(block) != horizon:
            _fail(
                "FUTURE_LENGTH_MISMATCH",
                f"future table has {len(block)} row(s) for series {series_id!r}; exactly "
                f"prediction_length={horizon} are required.",
                series_id=str(series_id),
                n_rows=len(block),
                expected=horizon,
            )
        origin = last_seen.loc[series_id]
        expected = pd.date_range(
            start=origin + frequency, periods=horizon, freq=frequency
        )
        observed = pd.DatetimeIndex(block[config.timestamp_column])
        if not observed.equals(expected):
            _fail(
                "FUTURE_TIMESTAMP_MISALIGNED",
                f"future timestamps for series {series_id!r} must start at the forecast "
                f"origin {origin + frequency} and follow the validated frequency "
                f"{frequency} exactly.",
                series_id=str(series_id),
                expected_first=str(expected[0]),
                observed_first=str(observed[0]) if len(observed) else None,
                expected_last=str(expected[-1]),
                observed_last=str(observed[-1]) if len(observed) else None,
            )

    return future, known_future


# --------------------------------------------------------------------------
# Entry point
# --------------------------------------------------------------------------


def validate_forecast_request(
    history_df: pd.DataFrame,
    config: ForecastConfig,
    *,
    trained_quantiles: tuple[float, ...] | list[float],
    model_context_length: int,
    model_prediction_length: int,
    future_df: pd.DataFrame | None = None,
    limits: ResourceLimits = DEFAULT_LIMITS,
    target_policy: str = "strict",
    allow_unroll: bool = False,
) -> ValidationResult:
    """Run RFC validation rules 1-21 and return the normalised request.

    Raises
    ------
    ValidationError
        On the first rule that fails, carrying a stable ``code``.
    """
    if not isinstance(history_df, pd.DataFrame):
        _fail(
            "HISTORY_NOT_A_DATAFRAME",
            f"history must be a pandas DataFrame, got {type(history_df).__name__}.",
        )

    targets = config.target_names

    # Rule 1 -----------------------------------------------------------------
    _require_columns(
        history_df,
        [config.id_column, config.timestamp_column, *targets],
        "MISSING_COLUMNS",
        "historical",
    )

    # Rule 11 (non-empty) ----------------------------------------------------
    if len(history_df) == 0:
        _fail("EMPTY_HISTORY", "historical table is empty.", n_rows=0)
    if len(history_df) > limits.max_rows:
        _fail(
            "RESOURCE_LIMIT",
            f"historical table has {len(history_df)} rows; the limit is {limits.max_rows}.",
            guard="max_rows",
            observed=len(history_df),
            limit=limits.max_rows,
        )

    history = history_df.copy()

    # Rule 2 -----------------------------------------------------------------
    history[config.timestamp_column] = _parse_timestamps(
        history, config.timestamp_column, "historical"
    )

    # Rule 3 -----------------------------------------------------------------
    if history[config.id_column].isna().any():
        _fail(
            "NULL_IDS",
            f"historical table has {int(history[config.id_column].isna().sum())} null "
            f"value(s) in id column {config.id_column!r}.",
            table="historical",
            column=config.id_column,
        )

    # Rule 4 and rule 19 -----------------------------------------------------
    history = _coerce_targets(history, targets, target_policy)

    # Rule 5 -----------------------------------------------------------------
    #: The frame is wide (one column per target), so a duplicated
    #: (id, timestamp) pair is a duplicated (id, timestamp, target_name)
    #: observation for every target at once.
    dup_mask = history.duplicated(subset=[config.id_column, config.timestamp_column], keep=False)
    if dup_mask.any():
        offenders = (
            history.loc[dup_mask, [config.id_column, config.timestamp_column]]
            .astype(str)
            .drop_duplicates()
            .head(5)
            .to_dict("records")
        )
        _fail(
            "DUPLICATE_OBSERVATIONS",
            f"historical table has {int(dup_mask.sum())} row(s) sharing an "
            f"(id, timestamp) pair; each (id, timestamp, target_name) observation must "
            f"be unique. Examples: {offenders}.",
            n_duplicate_rows=int(dup_mask.sum()),
            examples=offenders,
        )

    # Rule 6 -----------------------------------------------------------------
    history = history.sort_values(
        [config.id_column, config.timestamp_column], kind="mergesort"
    ).reset_index(drop=True)

    series_ids = list(pd.unique(history[config.id_column]))
    covariates = [c for c in history.columns if c not in config.reserved_columns]

    # Rule 20 ----------------------------------------------------------------
    for guard, observed, limit in (
        ("max_ids", len(series_ids), limits.max_ids),
        ("max_targets", len(targets), limits.max_targets),
        ("max_covariates", len(covariates), limits.max_covariates),
    ):
        if observed > limit:
            _fail(
                "RESOURCE_LIMIT",
                f"request exceeds the {guard} guard: {observed} > {limit}.",
                guard=guard,
                observed=observed,
                limit=limit,
            )

    #: Rules 4 and 19 for covariates. After the guards above so that a request
    #: with more covariates than the limit allows is still reported as a
    #: resource-limit breach rather than as whichever of them is first
    #: non-numeric.
    history = _coerce_covariates(history, covariates, target_policy, "historical")

    # Rules 7, 8, 9, 10 ------------------------------------------------------
    frequency, max_len, min_len = _shared_frequency(history, config, limits)
    confirmed_frequency = _check_declared_frequency(config.frequency, frequency)

    # Rule 11 (minimum context) and rule 20 (context guard) -------------------
    if config.context_length is not None:
        if config.context_length > limits.max_context_length:
            _fail(
                "RESOURCE_LIMIT",
                f"context_length {config.context_length} exceeds the DIMER guard "
                f"{limits.max_context_length}.",
                guard="max_context_length",
                observed=config.context_length,
                limit=limits.max_context_length,
            )
        if config.context_length < limits.min_observations:
            _fail(
                "CONTEXT_TOO_SHORT",
                f"context_length {config.context_length} is below the minimum of "
                f"{limits.min_observations} observations.",
                context_length=config.context_length,
                minimum=limits.min_observations,
            )

    # Rule 12 ----------------------------------------------------------------
    if config.prediction_length > limits.max_prediction_length:
        _fail(
            "PREDICTION_LENGTH_LIMIT",
            f"prediction_length {config.prediction_length} exceeds the DIMER guard "
            f"{limits.max_prediction_length}.",
            guard="max_prediction_length",
            observed=config.prediction_length,
            limit=limits.max_prediction_length,
        )
    if config.prediction_length > model_prediction_length and not allow_unroll:
        _fail(
            "PREDICTION_LENGTH_EXCEEDS_MODEL",
            f"prediction_length {config.prediction_length} exceeds the model's native "
            f"prediction length {model_prediction_length}. Upstream would satisfy this by "
            f"autoregressive unrolling, which v1 does not enable by default; pass "
            f"allow_unroll=True to opt in and the flag will be recorded in provenance.",
            requested=config.prediction_length,
            model_prediction_length=model_prediction_length,
        )

    # Rules 13 and 14 --------------------------------------------------------
    #: Hard-fail, with no opt-out. There was an ``allow_out_of_grid`` escape
    #: hatch; it exported a column named for the *requested* level while holding
    #: the substituted one, and recorded ``effective == requested`` in
    #: provenance, so the substitution the RFC exists to surface was invisible
    #: in the export (review R-2). Phase 1 has no consumer for it.
    out_of_grid = quantiles_in_grid(config.quantile_levels, trained_quantiles)
    if out_of_grid:
        hints = {
            str(q): nearest_grid_level(q, trained_quantiles)
            for q in out_of_grid
            if nearest_grid_level(q, trained_quantiles) is not None
        }
        hint = (
            f" Level(s) {list(hints)} are within 1e-6 of grid member(s) "
            f"{list(hints.values())}; membership is exact, so pass the grid value itself."
            if hints
            else ""
        )
        _fail(
            "QUANTILE_NOT_IN_GRID",
            f"requested quantile level(s) {out_of_grid} are outside the grid the pinned "
            f"model was trained on. Upstream would silently substitute the nearest trained "
            f"level, producing a column labelled with a level it does not contain, so this "
            f"request is rejected (RFC C-5).{hint}",
            out_of_grid=out_of_grid,
            trained_quantiles=[float(q) for q in trained_quantiles],
            nearest_grid_levels=hints,
        )

    # Rule 21 ----------------------------------------------------------------
    required_batch = len(targets) + len(covariates)
    if config.batch_size < required_batch:
        _fail(
            "BATCH_SIZE_TOO_SMALL",
            f"batch_size {config.batch_size} is below n_targets + n_covariates = "
            f"{len(targets)} + {len(covariates)} = {required_batch}. Upstream batches count "
            f"every series including covariates (RFC C-9).",
            batch_size=config.batch_size,
            n_targets=len(targets),
            n_covariates=len(covariates),
            required=required_batch,
        )

    # Rules 15-18 ------------------------------------------------------------
    future = None
    known_future_covariates: list[str] = []
    if future_df is not None:
        future, known_future_covariates = _validate_future(
            future_df, history, config, frequency, series_ids, covariates, target_policy
        )

    _ = model_context_length  # recorded by provenance; no DIMER-side rule needs it here

    return ValidationResult(
        history=history,
        future=future,
        frequency=frequency,
        series_ids=series_ids,
        target_names=list(targets),
        covariate_names=covariates,
        n_rows=len(history),
        max_series_length=max_len,
        min_series_length=min_len,
        requested_quantiles=[float(q) for q in config.quantile_levels],
        known_future_covariate_names=known_future_covariates,
        past_covariate_names=[c for c in covariates if c not in known_future_covariates],
        confirmed_frequency=confirmed_frequency,
    )


# --------------------------------------------------------------------------
# Role stage: validation (DIMER NOTEBOOK_SPEC 1.1 DAT24)
#
# `validate_inputs` is the public validation stage a tutorial calls before the
# model runs. It does not add a second rule set: it routes the request through
# `validate_forecast_request` with exactly the arguments
# `chronos2_pipeline.inference.forecast` passes, so it raises exactly what the
# forecast call would raise, and then reports what was proven as a
# machine-readable input manifest.
# --------------------------------------------------------------------------

#: The request contract and every named ceiling, in one readable structure.
INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "long-format pandas DataFrame: config.id_column, config.timestamp_column, one column "
        "per target name, and any further numeric column as a covariate"
    ),
    "timestamps": (
        "parseable, strictly regular and contiguous per series at one fixed-width frequency; "
        "monthly/quarterly/yearly/business-day and gappy calendars are refused"
    ),
    "targets": "finite numeric values; duplicate (id, timestamp) observations are refused",
    "min_observations_per_series": MIN_OBSERVATIONS,
    "max_ids": DEFAULT_LIMITS.max_ids,
    "max_targets": DEFAULT_LIMITS.max_targets,
    "max_covariates": DEFAULT_LIMITS.max_covariates,
    "max_rows": DEFAULT_LIMITS.max_rows,
    "max_context_length": DEFAULT_LIMITS.max_context_length,
    "max_prediction_length": DEFAULT_LIMITS.max_prediction_length,
    "quantile_levels": "must be members of the grid the pinned model was trained on, exactly",
    "future_df": (
        "known-future covariates only: id and timestamp columns plus covariates also present in "
        "the history; target columns are refused as leakage"
    ),
}


def validate_inputs(
    history_df: pd.DataFrame,
    config: ForecastConfig,
    model: Any,
    future_df: pd.DataFrame | None = None,
    *,
    limits: ResourceLimits = DEFAULT_LIMITS,
    allow_unroll: bool = False,
    target_policy: str = "strict",
    names: list[Any] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-series observations, verdict).

    Rejection is reported by raising exactly as :func:`chronos2_pipeline.inference.forecast`
    would — both go through :func:`validate_forecast_request` with the same arguments — so a
    caller that wants the finding recorded catches :class:`ValidationError` and stores
    ``str(exc)`` under ``findings``.

    ``model`` is the :class:`chronos2_pipeline.model.LoadedModel`; only its
    ``trained_quantiles``, ``model_context_length`` and ``model_prediction_length`` are read,
    because those are the model-side facts the rules need.
    """
    validated = validate_forecast_request(
        history_df,
        config,
        trained_quantiles=model.trained_quantiles,
        model_context_length=model.model_context_length,
        model_prediction_length=model.model_prediction_length,
        future_df=future_df,
        limits=limits,
        target_policy=target_policy,
        allow_unroll=allow_unroll,
    )
    if names is not None and len(names) != len(validated.series_ids):
        raise ValidationError(
            "NAMES_LENGTH_MISMATCH",
            f"names must have one entry per series: got {len(names)} for "
            f"{len(validated.series_ids)} series.",
            {"n_names": len(names), "n_series": len(validated.series_ids)},
        )
    history = validated.history
    inputs: list[dict[str, Any]] = []
    for index, series_id in enumerate(validated.series_ids):
        block = history[history[config.id_column] == series_id]
        timestamps = block[config.timestamp_column]
        inputs.append(
            {
                "id": names[index] if names else str(series_id),
                "series_id": str(series_id),
                "n_observations": int(len(block)),
                "first_timestamp": str(timestamps.min()),
                "last_timestamp": str(timestamps.max()),
                "targets": list(validated.target_names),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "n_rows": int(validated.n_rows),
        "prediction_length": int(config.prediction_length),
        "requested_quantile_levels": list(validated.requested_quantiles),
        "observed_frequency": str(validated.frequency),
        "confirmed_frequency": validated.confirmed_frequency,
        "longest_series_length": int(validated.max_series_length),
        "shortest_series_length": int(validated.min_series_length),
        "past_covariate_names": list(validated.past_covariate_names),
        "known_future_covariate_names": list(validated.known_future_covariate_names),
        "verdict": "accepted",
        "findings": [],
        "model_id": model.identity.model_id,
        "model_revision": model.identity.revision,
    }

**Module 6/7:** `src/chronos2_pipeline/inference.py` (carried verbatim; see the note above)

In [ ]:
"""Forecast entry point: validate, predict, normalise, record.

Covers all four RFC forecasting modes: A (univariate), B (multiple independent
series), C (multivariate / multi-target) and D (covariate-informed, past-only
and known-future). Phase 2 lifted the multi-target and covariate refusals that
Phase 1 carried; the validation layer already implemented the rules for both.

**Numeric covariates only.** See :func:`chronos2_pipeline.validation
._coerce_covariates` for why a categorical covariate is refused rather than
handed to an encoder whose route depends on the number of targets.

Five upstream behaviours are asserted on every call rather than trusted:

* ``predict_df`` returns the columns the rename map expects;
* it returns one row per (series, target, step), in that order — so a forecast
  cannot be exported under the wrong series or target label;
* the returned point and quantile values are finite;
* when ``0.5`` is requested, ``predictions`` is *exactly* the ``"0.5"`` column;
* every exported ``q<level>`` column names a level the loaded model was actually
  trained on, so no substituted quantile can leave under a borrowed label.

The second matters only once there is more than one target, and it is the
multi-target half of RFC C-1: upstream lays the frame out as ``target_name =
np.tile(np.repeat(target, prediction_length), n_inputs)``
(``chronos/chronos2/pipeline.py`` L955 in 2.3.1) and the values arrive in that
same order from a ravelled array. Nothing in the frame ties a row's number back
to the series it came from, so a change to either order would relabel forecasts
silently rather than fail.

The fourth is the median oracle (RFC C-2). Upstream 2.3.1 computes it that way
at ``chronos/chronos2/pipeline.py`` L816-818 — ``# NOTE: the median is returned
as the mean here`` — while its own docstrings call the value a mean. If a future
release makes it an actual mean, this assertion fails loudly instead of
relabelling a different statistic as ``prediction``.
"""

from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import UpstreamContractError` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import LoadedModel` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .provenance import build_provenance` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .validation import DEFAULT_LIMITS, ResourceLimits, validate_forecast_request` removed — names are kernel globals defined by the carried modules

__all__ = [
    "ForecastResult",
    "NORMALIZED_ID_COLUMN",
    "NORMALIZED_TIMESTAMP_COLUMN",
    "NORMALIZED_TARGET_COLUMN",
    "NORMALIZED_POINT_COLUMN",
    "RAW_POINT_COLUMN",
    "build_rename_map",
    "normalized_columns",
    "quantile_column_name",
    "forecast",
]

NORMALIZED_ID_COLUMN = "series_id"
NORMALIZED_TIMESTAMP_COLUMN = "timestamp"
NORMALIZED_TARGET_COLUMN = "target_name"
NORMALIZED_POINT_COLUMN = "prediction"
RAW_POINT_COLUMN = "predictions"


def quantile_column_name(level: float) -> str:
    """``0.1 -> "q0.1"``. Upstream names the raw column ``str(level)``."""
    return f"q{level}"


def build_rename_map(config: ForecastConfig) -> dict[str, str]:
    """The deterministic upstream-raw to DIMER-normalised column map.

    ``target_name`` is deliberately absent: it is carried through unchanged, and
    a rename entry would imply it is optional.
    """
    mapping = {
        config.id_column: NORMALIZED_ID_COLUMN,
        config.timestamp_column: NORMALIZED_TIMESTAMP_COLUMN,
        RAW_POINT_COLUMN: NORMALIZED_POINT_COLUMN,
    }
    for level in config.quantile_levels:
        mapping[str(level)] = quantile_column_name(level)
    return mapping


def normalized_columns(config: ForecastConfig) -> list[str]:
    """Exact column order of the normalised forecast frame."""
    return [
        NORMALIZED_ID_COLUMN,
        NORMALIZED_TIMESTAMP_COLUMN,
        NORMALIZED_TARGET_COLUMN,
        NORMALIZED_POINT_COLUMN,
        *(quantile_column_name(q) for q in config.quantile_levels),
    ]


@dataclass(frozen=True)
class ForecastResult:
    """A normalised forecast, the raw upstream frame, and full provenance."""

    forecast: pd.DataFrame
    raw: pd.DataFrame
    provenance: dict[str, Any]

    @property
    def model(self) -> dict[str, Any]:
        return self.provenance["model"]

    @property
    def inference(self) -> dict[str, Any]:
        return self.provenance["inference"]


def _assert_raw_contract(raw: pd.DataFrame, config: ForecastConfig) -> None:
    expected = [
        config.id_column,
        config.timestamp_column,
        NORMALIZED_TARGET_COLUMN,
        RAW_POINT_COLUMN,
        *(str(q) for q in config.quantile_levels),
    ]
    missing = [c for c in expected if c not in raw.columns]
    if missing:
        raise UpstreamContractError(
            f"predict_df returned a frame missing expected column(s) {missing}. "
            f"Present: {list(raw.columns)}. The pinned upstream contract recorded in "
            f"docs/rfc/0001-chronos-2.md no longer holds."
        )


def _assert_row_layout(
    raw: pd.DataFrame, config: ForecastConfig, series_ids: list[Any], horizon: int
) -> None:
    """One row per (series, target, step), in that order.

    Upstream builds the frame by ravelling a ``[n_tasks, n_variates, horizon]``
    array against row labels it generates separately — ``target_name`` from
    ``np.tile(np.repeat(target, prediction_length), n_inputs)`` and the id/
    timestamp block from ``future.iloc[np.repeat(item_rows, n_variates)]``
    (``chronos/chronos2/pipeline.py`` L951-957 in 2.3.1). The labels and the
    values are therefore aligned by construction and by nothing else: if a
    future release reorders either, every forecast still arrives, each one
    attached to the wrong series or the wrong target, and no column in the frame
    would contradict it.

    Series order is upstream's ``PreparedInput`` order, "the order in which
    item_ids first appear in df" (``preprocess.py`` L164). The validated history
    is sorted by id, so first appearance is that sorted order — the same list
    :func:`validate_forecast_request` returns.
    """
    targets = config.target_names
    expected_rows = len(series_ids) * len(targets) * horizon
    if len(raw) != expected_rows:
        raise UpstreamContractError(
            f"predict_df returned {len(raw)} row(s); the pinned contract is one row per "
            f"(series, target, step) = {len(series_ids)} x {len(targets)} x {horizon} = "
            f"{expected_rows}. The recorded upstream layout no longer holds, and rows "
            f"cannot be attributed to a series and target by position."
        )

    expected_targets = np.tile(np.repeat(targets, horizon), len(series_ids))
    observed_targets = raw[NORMALIZED_TARGET_COLUMN].to_numpy()
    if not np.array_equal(observed_targets, expected_targets):
        first = int(np.argmax(observed_targets != expected_targets))
        raise UpstreamContractError(
            f"predict_df returned target_name in an unexpected order: row {first} is "
            f"{observed_targets[first]!r}, the pinned layout puts "
            f"{expected_targets[first]!r} there. Values are aligned to these labels by "
            f"position alone, so a reordering would relabel forecasts silently."
        )

    expected_ids = np.repeat(np.asarray(series_ids, dtype=object), len(targets) * horizon)
    observed_ids = raw[config.id_column].to_numpy(dtype=object)
    if not np.array_equal(observed_ids, expected_ids):
        first = int(np.argmax(observed_ids != expected_ids))
        raise UpstreamContractError(
            f"predict_df returned series ids in an unexpected order: row {first} is "
            f"{observed_ids[first]!r}, the pinned layout puts {expected_ids[first]!r} "
            f"there. Values are aligned to these labels by position alone, so a "
            f"reordering would attribute forecasts to the wrong series."
        )


def _assert_finite(raw: pd.DataFrame, config: ForecastConfig) -> None:
    """A non-finite forecast is a distinct failure, and must be named as one.

    Checked before the median oracle: ``NaN == NaN`` is ``False``, so a
    NaN-valued output used to surface as "the median contract broke", sending a
    maintainer to the wrong upstream line for a condition that has nothing to do
    with the median (review R-9).
    """
    columns = [RAW_POINT_COLUMN, *(str(q) for q in config.quantile_levels)]
    offenders = {}
    for column in columns:
        values = pd.to_numeric(raw[column], errors="coerce").to_numpy(dtype=float)
        n_bad = int((~np.isfinite(values)).sum())
        if n_bad:
            offenders[column] = n_bad
    if offenders:
        raise UpstreamContractError(
            f"predict_df returned non-finite values (NaN/inf) in column(s) {offenders}. "
            f"A forecast frame containing NaN is not a forecast; v1 refuses it rather "
            f"than exporting it."
        )


def _assert_median_oracle(raw: pd.DataFrame, config: ForecastConfig) -> None:
    if 0.5 not in [float(q) for q in config.quantile_levels]:
        return
    point = raw[RAW_POINT_COLUMN].to_numpy()
    median = raw["0.5"].to_numpy()
    if point.shape != median.shape or not np.array_equal(point, median, equal_nan=True):
        n_diff = int((point != median).sum()) if point.shape == median.shape else -1
        raise UpstreamContractError(
            f"Upstream `predictions` is no longer exactly the 0.5 quantile "
            f"({n_diff} differing row(s)). This pipeline labels the point forecast "
            f"`prediction` on the recorded basis that it is the median; that basis "
            f"has changed and the label would now be wrong."
        )


def _assert_quantile_labels(
    config: ForecastConfig, trained_quantiles: tuple[float, ...] | list[float]
) -> None:
    """No ``q<level>`` column may be exported for a level the model cannot produce.

    Validation already hard-fails an out-of-grid request, so this can only fire
    if that gate is ever weakened or bypassed. It is here because the failure it
    guards against is silent: upstream substitutes the nearest trained level and
    returns it under the requested name, so a mislabelled column looks exactly
    like a correct one (RFC C-5, review R-2).
    """
    grid = {float(q) for q in trained_quantiles}
    mislabelled = [float(q) for q in config.quantile_levels if float(q) not in grid]
    if mislabelled:
        raise UpstreamContractError(
            f"Refusing to export quantile column(s) "
            f"{[quantile_column_name(q) for q in mislabelled]}: level(s) {mislabelled} "
            f"are not members of the trained grid {sorted(grid)}, so upstream can only "
            f"have substituted a different level under that name."
        )


def forecast(
    history_df: pd.DataFrame,
    config: ForecastConfig,
    model: LoadedModel,
    future_df: pd.DataFrame | None = None,
    *,
    limits: ResourceLimits = DEFAULT_LIMITS,
    allow_unroll: bool = False,
    target_policy: str = "strict",
    measure_latency: bool = False,
) -> ForecastResult:
    """Forecast one or more targets across one or more series.

    Parameters
    ----------
    history_df
        Long-format history: id column, timestamp column, one column per target.
        Every remaining column is a covariate — past-only unless it also appears
        in ``future_df``. Covariates must be numeric.
    config
        User parameters. ``config.target`` names one column or several.
    model
        The result of :func:`chronos2_pipeline.model.load_pinned_model`.
    future_df
        Known-future covariate values: the id and timestamp columns, plus at
        least one covariate that also exists in ``history_df``, with exactly
        ``prediction_length`` rows per series starting at the forecast origin.
        Target columns here are refused as leakage.
    allow_unroll
        Permit ``prediction_length`` beyond the model's native horizon, which
        upstream satisfies by autoregressive unrolling. Off by default; when on,
        ``autoregressive_unrolled`` is recorded in provenance.
    measure_latency
        Run one discarded warm-up ``predict_df`` before the scored call. Doubles
        the cost; off by default, in which case ``latency_seconds`` is a cold
        measurement and ``warm_up_performed`` is ``False``.

    Raises
    ------
    ValidationError
        Any of RFC rules 1-21.
    UpstreamContractError
        ``predict_df`` no longer matches the recorded pinned contract.
    """
    validated = validate_forecast_request(
        history_df,
        config,
        trained_quantiles=model.trained_quantiles,
        model_context_length=model.model_context_length,
        model_prediction_length=model.model_prediction_length,
        future_df=future_df,
        limits=limits,
        target_policy=target_policy,
        allow_unroll=allow_unroll,
    )

    predict_kwargs: dict[str, Any] = {
        "future_df": validated.future,
        "id_column": config.id_column,
        "timestamp_column": config.timestamp_column,
        #: Always a list, even for one target. Upstream wraps a bare string in a
        #: list anyway (``pipeline.py`` L900-901), so passing one is the same
        #: request with the shape stated rather than inferred.
        "target": list(config.target_names),
        "prediction_length": config.prediction_length,
        "quantile_levels": [float(q) for q in config.quantile_levels],
        "batch_size": config.batch_size,
        "context_length": config.context_length,
        "cross_learning": config.cross_learning,
        "validate_inputs": True,
        #: Only an alias validation affirmatively confirmed against the observed
        #: interval, never the raw request: upstream uses ``freq`` as-is to lay
        #: out the horizon and does not check it against the data (review R-1).
        "freq": validated.confirmed_frequency,
    }

    if measure_latency:
        model.pipeline.predict_df(validated.history, **predict_kwargs)

    started = time.perf_counter()
    raw = model.pipeline.predict_df(validated.history, **predict_kwargs)
    latency_seconds = time.perf_counter() - started

    _assert_raw_contract(raw, config)
    _assert_row_layout(raw, config, validated.series_ids, config.prediction_length)
    _assert_finite(raw, config)
    _assert_median_oracle(raw, config)
    _assert_quantile_labels(config, model.trained_quantiles)

    rename_map = build_rename_map(config)
    normalized = raw.rename(columns=rename_map)[normalized_columns(config)].reset_index(drop=True)

    model_ctx = model.model_context_length
    requested_ctx = config.context_length
    #: The context bound actually in force. ``min(requested, model)`` alone reported a
    #: ceiling rather than an effective value: a 100-observation series with
    #: ``context_length=None`` exported the model's full context length, a number larger
    #: than the history that existed. Upstream can only consume what a series holds, so
    #: the longest series caps it too (RFC C-6). Series shorter than this contributed
    #: less, which is why both observed lengths are exported alongside it.
    effective_ctx = min(requested_ctx or model_ctx, model_ctx, validated.max_series_length)
    effective_horizon = config.prediction_length

    provenance = build_provenance(
        model.identity,
        device=model.device,
        dtype=model.dtype,
        n_ids=validated.n_ids,
        n_targets=validated.n_targets,
        n_covariates=validated.n_covariates,
        past_covariate_names=list(validated.past_covariate_names),
        known_future_covariate_names=list(validated.known_future_covariate_names),
        requested_context_length=requested_ctx,
        effective_context_length=effective_ctx,
        longest_series_length=validated.max_series_length,
        shortest_series_length=validated.min_series_length,
        requested_prediction_length=config.prediction_length,
        effective_prediction_length=effective_horizon,
        autoregressive_unrolled=effective_horizon > model.model_prediction_length,
        requested_quantiles=validated.requested_quantiles,
        effective_quantiles=validated.requested_quantiles,
        batch_size=config.batch_size,
        cross_learning=config.cross_learning,
        latency_seconds=latency_seconds,
        warm_up_performed=measure_latency,
        frequency=config.frequency if config.frequency is not None else "",
        observed_frequency=str(validated.frequency),
    )

    return ForecastResult(forecast=normalized, raw=raw, provenance=provenance)

**Module 7/7:** `src/chronos2_pipeline/evaluation.py` (carried verbatim; see the note above)

In [ ]:
"""Chronological, leakage-safe evaluation and naive baselines.

This module deliberately operates on already-normalised forecast frames and on
held-out truth. It never trains or refits the model. The only split helper is
chronological: the last ``horizon`` timestamps of each series are removed from
history before inference, so future targets cannot leak into model context.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd

# standalone rewrite (build_notebook.py): `from .config import ForecastConfig` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .errors import ValidationError` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .inference import (` removed — names are kernel globals defined by the carried modules

__all__ = [
    "EvaluationResult",
    "HoldoutSplit",
    "chronological_holdout",
    "evaluate_forecast",
    "evaluation_report",
    "seasonal_naive_baseline",
    "last_value_baseline",
]


@dataclass(frozen=True)
class HoldoutSplit:
    """History and future truth produced by a chronological tail split."""

    history: pd.DataFrame
    truth: pd.DataFrame
    horizon: int


@dataclass(frozen=True)
class EvaluationResult:
    """Aligned row-level evidence plus per-series and aggregate metrics."""

    aligned: pd.DataFrame
    per_series: pd.DataFrame
    aggregate: dict[str, Any]
    quantiles: pd.DataFrame


def _fail(code: str, message: str, **details: Any) -> None:
    raise ValidationError(code, message, details)


def _prepared(frame: pd.DataFrame, config: ForecastConfig, table: str) -> pd.DataFrame:
    required = [config.id_column, config.timestamp_column, *config.target_names]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        _fail(
            "EVALUATION_MISSING_COLUMNS",
            f"{table} is missing required column(s) {missing}.",
            table=table,
            missing=missing,
        )
    out = frame.copy()
    out[config.timestamp_column] = pd.to_datetime(out[config.timestamp_column], errors="coerce")
    if out[config.timestamp_column].isna().any():
        _fail(
            "EVALUATION_BAD_TIMESTAMP",
            f"{table} contains timestamp values that cannot be parsed.",
            table=table,
        )
    if out[config.id_column].isna().any():
        _fail("EVALUATION_NULL_ID", f"{table} contains null series ids.", table=table)
    if out.duplicated([config.id_column, config.timestamp_column]).any():
        _fail(
            "EVALUATION_DUPLICATE_TIMESTAMP",
            f"{table} contains duplicate (id, timestamp) rows.",
            table=table,
        )
    for target in config.target_names:
        values = pd.to_numeric(out[target], errors="coerce")
        if values.isna().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
            _fail(
                "EVALUATION_BAD_TARGET",
                f"{table} target {target!r} contains missing or non-finite values.",
                table=table,
                target=target,
            )
        out[target] = values.astype(float)
    return out.sort_values(
        [config.id_column, config.timestamp_column], kind="mergesort"
    ).reset_index(drop=True)


def chronological_holdout(
    frame: pd.DataFrame,
    config: ForecastConfig,
    *,
    horizon: int | None = None,
    minimum_context: int = 3,
) -> HoldoutSplit:
    """Remove the last ``horizon`` timestamps of every series as future truth.

    No random split helper exists in the public API. Every target value in the
    returned ``truth`` frame is absent from ``history``.
    """

    data = _prepared(frame, config, "evaluation source")
    holdout = config.prediction_length if horizon is None else horizon
    if isinstance(holdout, bool) or not isinstance(holdout, int) or holdout < 1:
        _fail(
            "EVALUATION_INVALID_HORIZON",
            f"horizon must be a positive integer, got {holdout!r}.",
            horizon=holdout,
        )
    if (
        isinstance(minimum_context, bool)
        or not isinstance(minimum_context, int)
        or minimum_context < 1
    ):
        _fail(
            "EVALUATION_INVALID_MINIMUM_CONTEXT",
            f"minimum_context must be a positive integer, got {minimum_context!r}.",
            minimum_context=minimum_context,
        )

    history_parts: list[pd.DataFrame] = []
    truth_parts: list[pd.DataFrame] = []
    for series_id, block in data.groupby(config.id_column, sort=False):
        if len(block) < holdout + minimum_context:
            _fail(
                "EVALUATION_SERIES_TOO_SHORT",
                f"series {series_id!r} has {len(block)} row(s); need at least "
                f"{holdout + minimum_context} for a {holdout}-step holdout with "
                f"{minimum_context} context rows.",
                series_id=series_id,
                n_rows=len(block),
                horizon=holdout,
                minimum_context=minimum_context,
            )
        history_parts.append(block.iloc[:-holdout].copy())
        truth_parts.append(block.iloc[-holdout:].copy())

    history = pd.concat(history_parts, ignore_index=True)
    truth = pd.concat(truth_parts, ignore_index=True)

    for series_id in history[config.id_column].drop_duplicates().tolist():
        h = history[history[config.id_column] == series_id]
        t = truth[truth[config.id_column] == series_id]
        if not h[config.timestamp_column].max() < t[config.timestamp_column].min():
            _fail(
                "EVALUATION_HOLDOUT_OVERLAP",
                f"chronological split for series {series_id!r} overlaps the held-out future.",
                series_id=series_id,
            )

    return HoldoutSplit(history=history, truth=truth, horizon=holdout)


def _truth_long(truth_df: pd.DataFrame, config: ForecastConfig) -> pd.DataFrame:
    truth = _prepared(truth_df, config, "truth")
    long = truth.melt(
        id_vars=[config.id_column, config.timestamp_column],
        value_vars=config.target_names,
        var_name=NORMALIZED_TARGET_COLUMN,
        value_name="truth",
    ).rename(
        columns={
            config.id_column: NORMALIZED_ID_COLUMN,
            config.timestamp_column: NORMALIZED_TIMESTAMP_COLUMN,
        }
    )
    return long[
        [
            NORMALIZED_ID_COLUMN,
            NORMALIZED_TIMESTAMP_COLUMN,
            NORMALIZED_TARGET_COLUMN,
            "truth",
        ]
    ].sort_values(
        [NORMALIZED_ID_COLUMN, NORMALIZED_TARGET_COLUMN, NORMALIZED_TIMESTAMP_COLUMN],
        kind="mergesort",
    ).reset_index(drop=True)


def _normalised_forecast(forecast_df: pd.DataFrame, config: ForecastConfig) -> pd.DataFrame:
    required = [
        NORMALIZED_ID_COLUMN,
        NORMALIZED_TIMESTAMP_COLUMN,
        NORMALIZED_TARGET_COLUMN,
        NORMALIZED_POINT_COLUMN,
    ]
    missing = [column for column in required if column not in forecast_df.columns]
    if missing:
        _fail(
            "EVALUATION_FORECAST_SCHEMA",
            f"forecast is missing required normalised column(s) {missing}.",
            missing=missing,
        )
    out = forecast_df.copy()
    out[NORMALIZED_TIMESTAMP_COLUMN] = pd.to_datetime(
        out[NORMALIZED_TIMESTAMP_COLUMN], errors="coerce"
    )
    if out[NORMALIZED_TIMESTAMP_COLUMN].isna().any():
        _fail("EVALUATION_FORECAST_SCHEMA", "forecast contains unparseable timestamps.")
    if out.duplicated(
        [NORMALIZED_ID_COLUMN, NORMALIZED_TIMESTAMP_COLUMN, NORMALIZED_TARGET_COLUMN]
    ).any():
        _fail(
            "EVALUATION_FORECAST_SCHEMA",
            "forecast contains duplicate (series_id, timestamp, target_name) rows.",
        )
    point = pd.to_numeric(out[NORMALIZED_POINT_COLUMN], errors="coerce")
    if point.isna().any() or not np.isfinite(point.to_numpy(dtype=float)).all():
        _fail("EVALUATION_FORECAST_NONFINITE", "forecast point predictions must be finite.")
    out[NORMALIZED_POINT_COLUMN] = point.astype(float)

    for level in config.quantile_levels:
        column = quantile_column_name(level)
        if column in out.columns:
            values = pd.to_numeric(out[column], errors="coerce")
            if values.isna().any() or not np.isfinite(values.to_numpy(dtype=float)).all():
                _fail(
                    "EVALUATION_FORECAST_NONFINITE",
                    f"forecast quantile column {column!r} must be finite.",
                    column=column,
                )
            out[column] = values.astype(float)
    return out


def evaluate_forecast(
    forecast_df: pd.DataFrame,
    truth_df: pd.DataFrame,
    config: ForecastConfig,
) -> EvaluationResult:
    """Score a DIMER-normalised forecast against held-out truth.

    Point metrics are MAE/RMSE. Quantile metrics use pinball loss. If at least
    two requested quantile columns are present, empirical coverage is measured
    between the lowest and highest requested levels.
    """

    forecast = _normalised_forecast(forecast_df, config)
    truth = _truth_long(truth_df, config)
    keys = [NORMALIZED_ID_COLUMN, NORMALIZED_TIMESTAMP_COLUMN, NORMALIZED_TARGET_COLUMN]
    merged = truth.merge(forecast, on=keys, how="outer", indicator=True, validate="one_to_one")
    if not (merged["_merge"] == "both").all():
        missing_forecast = merged.loc[merged["_merge"] == "left_only", keys].to_dict("records")
        missing_truth = merged.loc[merged["_merge"] == "right_only", keys].to_dict("records")
        _fail(
            "EVALUATION_COVERAGE_MISMATCH",
            "forecast and truth do not cover exactly the same (series, timestamp, target) rows.",
            missing_forecast=missing_forecast[:10],
            missing_truth=missing_truth[:10],
        )
    merged = merged.drop(columns="_merge")
    merged["error"] = merged[NORMALIZED_POINT_COLUMN] - merged["truth"]
    merged["absolute_error"] = merged["error"].abs()
    merged["squared_error"] = merged["error"] ** 2

    per_series = (
        merged.groupby([NORMALIZED_ID_COLUMN, NORMALIZED_TARGET_COLUMN], sort=True)
        .agg(
            n=("truth", "size"),
            mae=("absolute_error", "mean"),
            mse=("squared_error", "mean"),
        )
        .reset_index()
    )
    per_series["rmse"] = np.sqrt(per_series.pop("mse"))
    per_series = per_series[
        [NORMALIZED_ID_COLUMN, NORMALIZED_TARGET_COLUMN, "n", "mae", "rmse"]
    ]

    aggregate: dict[str, Any] = {
        "n": int(len(merged)),
        "mae": float(merged["absolute_error"].mean()),
        "rmse": float(np.sqrt(merged["squared_error"].mean())),
    }

    quantile_rows: list[dict[str, Any]] = []
    present_levels: list[float] = []
    for level in config.quantile_levels:
        column = quantile_column_name(level)
        if column not in merged.columns:
            continue
        present_levels.append(float(level))
        residual = merged["truth"] - merged[column]
        loss = np.where(
            residual >= 0.0,
            float(level) * residual,
            (float(level) - 1.0) * residual,
        )
        quantile_rows.append(
            {
                "quantile": float(level),
                "n": int(len(merged)),
                "pinball_loss": float(np.mean(loss)),
            }
        )
    quantiles = pd.DataFrame(quantile_rows, columns=["quantile", "n", "pinball_loss"])

    if len(present_levels) >= 2:
        lower = min(present_levels)
        upper = max(present_levels)
        lower_col = quantile_column_name(lower)
        upper_col = quantile_column_name(upper)
        covered = (merged["truth"] >= merged[lower_col]) & (merged["truth"] <= merged[upper_col])
        aggregate.update(
            {
                "interval_lower_quantile": lower,
                "interval_upper_quantile": upper,
                "interval_coverage": float(covered.mean()),
            }
        )

    return EvaluationResult(
        aligned=merged.sort_values(keys, kind="mergesort").reset_index(drop=True),
        per_series=per_series,
        aggregate=aggregate,
        quantiles=quantiles,
    )


def last_value_baseline(
    history_df: pd.DataFrame,
    truth_df: pd.DataFrame,
    config: ForecastConfig,
) -> pd.DataFrame:
    """Repeat each series/target's final observed history value over the truth window."""

    history = _prepared(history_df, config, "history")
    truth = _truth_long(truth_df, config)
    rows: list[pd.DataFrame] = []
    for target in config.target_names:
        last = (
            history.sort_values([config.id_column, config.timestamp_column], kind="mergesort")
            .groupby(config.id_column, sort=False)[target]
            .last()
        )
        block = truth[truth[NORMALIZED_TARGET_COLUMN] == target].copy()
        block[NORMALIZED_POINT_COLUMN] = block[NORMALIZED_ID_COLUMN].map(last)
        if block[NORMALIZED_POINT_COLUMN].isna().any():
            _fail(
                "EVALUATION_BASELINE_MISSING_HISTORY",
                f"last-value baseline has no history for at least one series for "
                f"target {target!r}.",
                target=target,
            )
        rows.append(
            block[
                [
                    NORMALIZED_ID_COLUMN,
                    NORMALIZED_TIMESTAMP_COLUMN,
                    NORMALIZED_TARGET_COLUMN,
                    NORMALIZED_POINT_COLUMN,
                ]
            ]
        )
    return pd.concat(rows, ignore_index=True).sort_values(
        [NORMALIZED_ID_COLUMN, NORMALIZED_TARGET_COLUMN, NORMALIZED_TIMESTAMP_COLUMN],
        kind="mergesort",
    ).reset_index(drop=True)


def seasonal_naive_baseline(
    history_df: pd.DataFrame,
    truth_df: pd.DataFrame,
    config: ForecastConfig,
    *,
    season_length: int,
) -> pd.DataFrame:
    """Repeat the final explicit season over the truth window.

    ``season_length`` is intentionally required. The pipeline does not infer a
    seasonal period from timestamps and thereby smuggle an unreviewed heuristic
    into the benchmark.
    """

    if isinstance(season_length, bool) or not isinstance(season_length, int) or season_length < 1:
        _fail(
            "EVALUATION_INVALID_SEASON_LENGTH",
            f"season_length must be a positive integer, got {season_length!r}.",
            season_length=season_length,
        )
    history = _prepared(history_df, config, "history")
    truth = _truth_long(truth_df, config)
    rows: list[dict[str, Any]] = []
    for series_id, truth_series in truth.groupby(NORMALIZED_ID_COLUMN, sort=False):
        h = history[history[config.id_column] == series_id].sort_values(
            config.timestamp_column, kind="mergesort"
        )
        if len(h) < season_length:
            _fail(
                "EVALUATION_SEASON_TOO_LONG",
                f"series {series_id!r} has {len(h)} history row(s), fewer than "
                f"season_length={season_length}.",
                series_id=series_id,
                n_history=len(h),
                season_length=season_length,
            )
        for target in config.target_names:
            season = h[target].tail(season_length).to_numpy(dtype=float)
            target_truth = truth_series[
                truth_series[NORMALIZED_TARGET_COLUMN] == target
            ].sort_values(NORMALIZED_TIMESTAMP_COLUMN, kind="mergesort")
            for step, (_, truth_row) in enumerate(target_truth.iterrows()):
                rows.append(
                    {
                        NORMALIZED_ID_COLUMN: series_id,
                        NORMALIZED_TIMESTAMP_COLUMN: truth_row[NORMALIZED_TIMESTAMP_COLUMN],
                        NORMALIZED_TARGET_COLUMN: target,
                        NORMALIZED_POINT_COLUMN: float(season[step % season_length]),
                    }
                )
    return pd.DataFrame(rows).sort_values(
        [NORMALIZED_ID_COLUMN, NORMALIZED_TARGET_COLUMN, NORMALIZED_TIMESTAMP_COLUMN],
        kind="mergesort",
    ).reset_index(drop=True)


# --------------------------------------------------------------------------
# Role stage: evaluation (DIMER NOTEBOOK_SPEC 1.1 EVAL21)
#
# `evaluation_report` is the public evaluation stage a tutorial calls after the
# forecast. It adds no metric of its own: every number comes from
# `evaluate_forecast` (MAE, RMSE, per-quantile pinball loss, empirical interval
# coverage) and the two naive baselines above, so the metric ids in the report
# are this module's own helper names (EVAL2). Without held-out truth the report
# still exists and says what would make the task measurable (EVAL9).
# --------------------------------------------------------------------------


def _aggregate_metrics(evaluation: EvaluationResult, *, estimation: str) -> list[dict[str, Any]]:
    metrics: list[dict[str, Any]] = [
        {"id": "evaluate_forecast", "metric": name, "value": float(value), "estimation": estimation}
        for name, value in evaluation.aggregate.items()
        if name in ("mae", "rmse", "interval_coverage")
    ]
    for row in evaluation.quantiles.to_dict("records"):
        metrics.append(
            {
                "id": "evaluate_forecast",
                "metric": "pinball_loss",
                "quantile": float(row["quantile"]),
                "value": float(row["pinball_loss"]),
                "estimation": estimation,
            }
        )
    return metrics


def evaluation_report(
    result: Any,
    truth_df: pd.DataFrame | None = None,
    *,
    config: ForecastConfig,
    history_df: pd.DataFrame | None = None,
    season_length: int | None = None,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``result`` is the :class:`chronos2_pipeline.inference.ForecastResult` (or its
    ``forecast`` frame). With ``truth_df`` — the held-out future produced by
    :func:`chronological_holdout` or supplied from outside — the report carries the
    :func:`evaluate_forecast` metrics for the model and, when ``history_df`` is given, for
    :func:`last_value_baseline` (and :func:`seasonal_naive_baseline` when ``season_length``
    is given too) with the verdict ``sample-sanity``: one chronological tail split of one
    sample, no dispersion estimate. Without truth the verdict is ``not-measurable`` and the
    report says what data would make the task measurable. Everything ``evaluate_forecast``
    would raise (a coverage mismatch, a malformed frame) is raised here unchanged.
    """
    forecast_df = getattr(result, "forecast", result)
    provenance = getattr(result, "provenance", None) or {}
    model_block = provenance.get("model", {}) if isinstance(provenance, dict) else {}
    inference_block = provenance.get("inference", {}) if isinstance(provenance, dict) else {}
    base: dict[str, Any] = {
        "task": "zero-shot probabilistic time-series forecasting",
        "score_semantics": (
            "prediction is the median (q0.5) of the model's quantile forecast; the requested "
            "quantiles summarise the predictive distribution and are not calibrated intervals"
        ),
        "sample_kind": sample_kind,
        "horizon": int(config.prediction_length),
        "effective_context_length": inference_block.get("effective_context_length"),
        "n_forecast_rows": int(len(forecast_df)),
        "n_series": int(forecast_df[NORMALIZED_ID_COLUMN].nunique()),
        "targets": sorted(str(t) for t in forecast_df[NORMALIZED_TARGET_COLUMN].unique()),
        "model_id": model_block.get("name"),
        "model_revision": model_block.get("revision"),
    }
    if truth_df is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no held-out future truth was supplied for the forecast window",
            "needs": (
                "the true target values for the forecast window, aligned on (series_id, "
                "timestamp, target_name) - e.g. a chronological tail holdout of your own history "
                "(chronological_holdout) scored with evaluate_forecast against last_value_baseline"
            ),
        }
    estimation = "single chronological tail holdout of one sample; no dispersion estimate"
    evaluation = evaluate_forecast(forecast_df, truth_df, config)
    baselines: list[dict[str, Any]] = []
    if history_df is not None:
        last_value = evaluate_forecast(
            last_value_baseline(history_df, truth_df, config), truth_df, config
        )
        baselines.append(
            {
                "id": "last_value_baseline",
                "metrics": _aggregate_metrics(last_value, estimation=estimation),
            }
        )
        if season_length is not None:
            seasonal = evaluate_forecast(
                seasonal_naive_baseline(history_df, truth_df, config, season_length=season_length),
                truth_df,
                config,
            )
            baselines.append(
                {
                    "id": "seasonal_naive_baseline",
                    "season_length": int(season_length),
                    "metrics": _aggregate_metrics(seasonal, estimation=estimation),
                }
            )
    return {
        **base,
        "metrics": _aggregate_metrics(evaluation, estimation=estimation),
        "baselines": baselines,
        "verdict": "sample-sanity",
        "reason": (
            f"{int(evaluation.aggregate['n'])} held-out (series, timestamp, target) rows from one "
            f"chronological tail split of the tutorial sample; not a benchmark"
        ),
        "needs": (
            "rolling-origin backtests over several forecast windows of the deployment domain's own "
            "history, compared against the naive baselines, for any generalisable accuracy claim"
        ),
    }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `95a9710e2596…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `load_pinned_model(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "chronos-2",
  "modelId": "amazon/chronos-2",
  "revision": "95a9710e2596287d08352589f42634fa5abdf0a7",
  "files": [
    {
      "path": "README.md",
      "bytes": 31,
      "sha256": "4bcf87ecfbbb8e07a01b21415a970c8b53a5283bf6872b657040d3f45c9241f7"
    },
    {
      "path": "config.json",
      "bytes": 1067,
      "sha256": "ef1143bfdc9c0376d9a056eefca46cb4b1ec3d0ffacd541ff56feb40fb708031"
    },
    {
      "path": "model.safetensors",
      "bytes": 477930472,
      "sha256": "ddcda3c7508bf2528087723e98a20707cc04b7f370ae275a9fd88078ddba4f42"
    }
  ],
  "totalBytes": 477931570
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = load_pinned_model(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: one hourly series of 96 steps built from a fixed formula (a linear trend plus a 24-step and a 12-step sinusoid), rendered to CSV bytes exactly as the repository's `examples/sample-data/generate_samples.py` writes them, so its SHA-256 is asserted against the digest the repository checks in — the notebook proves it is forecasting the very bytes the repository tests. It is deterministic teaching data, not benchmark evidence. BYOD requires unique UTF-8 CSV headers plus `series_id`, `timestamp`, and `target`; duplicate headers are rejected **before pandas can rename them**. Additional numeric covariates are permitted by the pipeline. Set `USE_BYOD=True` in Colab (upload one CSV), or set `DIMER_BYOD_PATH` in automation. Look for the input source, its SHA-256 and the first rows.

In [ ]:
import csv
import hashlib
import io
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

USE_BYOD = False  # @param {type:"boolean"}
PREDICTION_LENGTH = 12  # @param {type:"integer"}
BYOD_PATH = os.environ.get("DIMER_BYOD_PATH")
SAMPLE_SHA256 = "eff96b1a9bec5a81aa4021b4d308c29e18fc89be0cf8eb54e755c2efe535c7f7"  # examples/sample-data/SHA256SUMS


def read_checked_csv(payload: bytes) -> pd.DataFrame:
    rows = csv.reader(io.StringIO(payload.decode("utf-8-sig")))
    try:
        header = next(rows)
    except StopIteration as exc:
        raise ValueError("CSV is empty.") from exc
    duplicates = sorted(name for name, count in Counter(header).items() if count > 1)
    if duplicates:
        raise ValueError(f"Duplicate CSV header(s) are ambiguous: {duplicates}")
    return pd.read_csv(io.BytesIO(payload))


def synthetic_univariate_csv() -> bytes:
    # The repository's generate_samples.py formula and canonical CSV rendering; no random state.
    n = 96
    step = np.arange(n, dtype=float)
    frame = pd.DataFrame(
        {
            "series_id": "A",
            "timestamp": pd.date_range("2026-01-01", periods=n, freq="h"),
            "target": 100.0 + 0.25 * step + 7.0 * np.sin(2.0 * np.pi * step / 24.0) + 1.5 * np.cos(2.0 * np.pi * step / 12.0),
        }
    )
    return frame.to_csv(index=False, date_format="%Y-%m-%dT%H:%M:%S", float_format="%.4f", lineterminator="\n").encode("utf-8")


if BYOD_PATH:
    payload = Path(BYOD_PATH).read_bytes()
    input_source = f"BYOD path: {BYOD_PATH}"
    sample_kind = "BYOD"
elif USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV.")
    name, payload = next(iter(uploaded.items()))
    input_source = f"BYOD upload: {name}"
    sample_kind = "BYOD"
else:
    payload = synthetic_univariate_csv()
    observed = hashlib.sha256(payload).hexdigest()
    if observed != SAMPLE_SHA256:
        raise ValueError(f"Synthetic sample digest mismatch: {observed} != {SAMPLE_SHA256}")
    input_source = "synthetic sample generated in code (== examples/sample-data/chronos_univariate.csv)"
    sample_kind = "synthetic"

frame = read_checked_csv(payload)
input_sha256 = hashlib.sha256(payload).hexdigest()
print({"sample_kind": sample_kind, "input_source": input_source, "input_sha256": input_sha256, "rows": len(frame), "columns": list(frame.columns)})
print(frame.head())

## 5. Chronological holdout, ceilings, validate → input manifest

The final `PREDICTION_LENGTH` timestamps of each series are held out as truth; future target values never enter model context (chronological holdout — **no random split** or training initialization). Before anything runs, the cell prints the operational ceilings: the pinned model exposes an 8,192-step context and native 1,024-step horizon, and the DIMER request guards (`ResourceLimits`) cap one request at 1,000 series IDs, 64 targets, 64 covariates, 5,000,000 rows, 8,192 context steps, and 4,096 forecast steps; horizons above 1,024 require explicit autoregressive unrolling. `runtime_versions` reports the installed stack. `validate_inputs` is the package's public validation stage: it routes the history through `validate_forecast_request` with exactly the arguments `forecast` passes, so it raises exactly what the forecast call would raise, and returns an **input manifest** naming the schema and ceilings, each series' observed span, the confirmed frequency and the verdict; it is written to `outputs/chronos_2_forecasting_input_manifest.json`. To show what rejection looks like, the cell also validates a deliberately broken copy of the history (one non-finite target) and records the pipeline's own error as a finding.

In [ ]:
import os

os.makedirs('outputs', exist_ok=True)
config = ForecastConfig(target="target", prediction_length=PREDICTION_LENGTH, quantile_levels=[0.1, 0.5, 0.9])
split = chronological_holdout(frame, config)
for sid in split.history[config.id_column].drop_duplicates():
    h = split.history[split.history[config.id_column] == sid]
    t = split.truth[split.truth[config.id_column] == sid]
    assert h[config.timestamp_column].max() < t[config.timestamp_column].min()

versions = runtime_versions()
print({"runtime_versions": versions})
print("DIMER request limits:", ResourceLimits())
print("Pinned-model context limit: 8192")
print("Pinned-model native prediction length: 1024")
print({"ceilings": {"min_observations_per_series": MIN_OBSERVATIONS, "max_ids": DEFAULT_LIMITS.max_ids, "max_targets": DEFAULT_LIMITS.max_targets, "max_covariates": DEFAULT_LIMITS.max_covariates, "max_rows": DEFAULT_LIMITS.max_rows, "max_context_length": DEFAULT_LIMITS.max_context_length, "max_prediction_length": DEFAULT_LIMITS.max_prediction_length, "model_context_length": pipe.model_context_length, "model_prediction_length": pipe.model_prediction_length}})

series_names = [str(sid) for sid in split.history[config.id_column].drop_duplicates()]
input_manifest = validate_inputs(split.history, config, pipe, names=series_names)
# Demonstrate rejection on a history that breaks a rule; the finding is recorded, not swallowed.
broken = split.history.copy()
broken.loc[broken.index[0], "target"] = float("nan")
try:
    validate_inputs(broken, config, pipe)
except ValidationError as exc:
    input_manifest["findings"].append({"input": "non-finite-target-probe", "verdict": "rejected", "code": exc.code, "message": str(exc)})
with open("outputs/chronos_2_forecasting_input_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False, default=str)
print(json.dumps(input_manifest, indent=2, default=str))

## 6. Forecast

`forecast(split.history, config, pipe)` is the package's public inference path: it validates again, calls the pinned `chronos-forecasting` package's `predict_df` on the verified snapshot (it does not request model-repository remote code), and normalizes the output. Normalized output fields are `series_id`, `timestamp`, `target_name`, `prediction`, and requested quantiles such as `q0.1`, `q0.5`, `q0.9`. `prediction` is the **median (`q0.5`)**, not a mean. Model quantiles summarize the predictive distribution; they are **not guaranteed frequentist confidence intervals** and are not assumed calibrated on a new domain — the pipeline ships no threshold and no calibration. The effective context and prediction lengths actually used are printed with the first forecast rows; floating-point details can vary across runtime/hardware builds and latency is run-dependent.

In [ ]:
result = forecast(split.history, config, pipe)
print(result.forecast.head())
print("effective_context_length:", result.inference["effective_context_length"])
print("effective_prediction_length:", result.inference["effective_prediction_length"])
print({"n_forecast_rows": len(result.forecast), "quantile_columns": [c for c in result.forecast.columns if c.startswith("q")], "latency_seconds": result.inference.get("latency_seconds")})

## 7. Evaluate → evaluation report

Metrics use the same chronological holdout: **MAE** is mean absolute error; **RMSE** weights larger misses more; **pinball loss** evaluates a quantile asymmetrically; **empirical interval coverage** is the fraction of held-out truths inside the requested outer quantiles. These are tutorial/sanity metrics, not benchmark evidence. `evaluation_report` is the package's public evaluation stage and always produces a report: with the held-out truth it carries the `evaluate_forecast` metrics for Chronos-2 and for the `last_value_baseline` (and the `seasonal_naive_baseline` when applicable) with the verdict `sample-sanity` — one chronological tail split of one sample, no dispersion estimate; without truth (a forecast of the real future) the verdict is `not-measurable` and the report states what data would make the task measurable. The seasonal-naive comparator runs only for uniformly hourly data when **every series has at least 24 post-holdout history rows**; short but otherwise valid hourly BYOD therefore skips this optional comparator instead of failing. The report is written to `outputs/chronos_2_forecasting_evaluation_report.json`.

In [ ]:
evaluation = evaluate_forecast(result.forecast, split.truth, config)
last_value = last_value_baseline(split.history, split.truth, config)
last_value_evaluation = evaluate_forecast(last_value, split.truth, config)

ordered = split.history.sort_values([config.id_column, config.timestamp_column])
steps = ordered.groupby(config.id_column)[config.timestamp_column].diff().dropna()
minimum_history = int(ordered.groupby(config.id_column, sort=False).size().min())
can_use_daily_seasonal = not steps.empty and (steps == pd.Timedelta(hours=1)).all() and minimum_history >= 24

seasonal_evaluation = None
if can_use_daily_seasonal:
    seasonal = seasonal_naive_baseline(split.history, split.truth, config, season_length=24)
    seasonal_evaluation = evaluate_forecast(seasonal, split.truth, config)
seasonal_metrics = None if seasonal_evaluation is None else seasonal_evaluation.aggregate

print("Chronos-2:", evaluation.aggregate)
print("Last-value:", last_value_evaluation.aggregate)
print("Seasonal-naive:", seasonal_metrics)
print("Quantile metrics:")
print(evaluation.quantiles)
print("Per-series metrics:")
print(evaluation.per_series)

report = evaluation_report(result, split.truth, config=config, history_df=split.history, season_length=24 if can_use_daily_seasonal else None, sample_kind=sample_kind)
with open("outputs/chronos_2_forecasting_evaluation_report.json", "w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False, default=str)
print(json.dumps(report, indent=2, default=str))
if report["verdict"] == "not-measurable":
    print("No held-out truth was supplied, so evaluate_forecast is not computed; the forecast above is sanity evidence only.")

## 8. Visualize the held-out forecast

For multi-series BYOD, this compact SVG deliberately visualizes one series and one target instead of interleaving unrelated lines. The machine-readable exports remain authoritative for all rows.

In [ ]:
from html import escape

plot_series = result.forecast["series_id"].iloc[0]
plot_target = result.forecast["target_name"].iloc[0]
plot_forecast = result.forecast[(result.forecast["series_id"] == plot_series) & (result.forecast["target_name"] == plot_target)].sort_values("timestamp")
plot_truth = split.truth[split.truth[config.id_column] == plot_series].sort_values(config.timestamp_column)

values = plot_forecast["q0.1"].tolist() + plot_forecast["prediction"].tolist() + plot_forecast["q0.9"].tolist() + plot_truth[plot_target].tolist()
low, high = min(values), max(values)
span = high - low or 1.0
width, height = 760, 280
left, right, top, bottom = 48, width - 20, 30, height - 38


def points(series):
    n_steps = max(len(series) - 1, 1)
    return " ".join(f"{left + (right - left) * i / n_steps:.1f},{bottom - (bottom - top) * (float(v) - low) / span:.1f}" for i, v in enumerate(series))


layers = [("q0.1", plot_forecast["q0.1"].tolist()), ("median", plot_forecast["prediction"].tolist()), ("q0.9", plot_forecast["q0.9"].tolist()), ("truth", plot_truth[plot_target].tolist())]
strokes = ["#111827", "#2563eb", "#dc2626", "#059669"]
svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">']
title = f"{escape(str(plot_series))} / {escape(str(plot_target))}: held-out future"
svg.append(f'<text x="{left}" y="18" font-family="sans-serif" font-size="14">{title}</text>')
for idx, (label, series) in enumerate(layers):
    stroke = strokes[idx]
    svg.append(f'<polyline fill="none" stroke="{stroke}" stroke-width="2" points="{points(series)}"/>')
    svg.append(f'<text x="{left + 120 * idx}" y="{height - 10}" font-family="sans-serif" font-size="12" fill="{stroke}">{escape(label)}</text>')
svg.append("</svg>")

svg_path = Path("outputs") / "chronos_forecast.svg"
svg_path.write_text("\n".join(svg), encoding="utf-8")
try:
    from IPython.display import SVG, display

    display(SVG(filename=str(svg_path)))
except ImportError:
    print("Forecast SVG written to", svg_path)

## 9. Primary Mode C — multi-target forecasting

Multi-target forecasting is a primary user-facing capability, so this path runs by default. For interface demonstration only, `target_aux` is deterministically derived from `target`; it is **not an independent benchmark variable**. The assertion proves the normalized output preserves both target names.

In [ ]:
mode_c_frame = frame.copy()
mode_c_frame["target_aux"] = 0.5 * pd.to_numeric(mode_c_frame["target"]) + 10.0
mode_c_config = ForecastConfig(target=["target", "target_aux"], prediction_length=PREDICTION_LENGTH, quantile_levels=[0.1, 0.5, 0.9])
mode_c_split = chronological_holdout(mode_c_frame, mode_c_config)
mode_c_result = forecast(mode_c_split.history, mode_c_config, pipe)
observed_targets = set(mode_c_result.forecast["target_name"].unique())
assert observed_targets == {"target", "target_aux"}
print("Mode C target names:", sorted(observed_targets))

## 10. Export outputs and provenance

Exports contain the normalized univariate forecast (`outputs/chronos_forecast.csv`), the Mode C multi-target forecast, the per-series evaluation, the aggregate/quantile metrics (`outputs/chronos_evaluation.json`), the pipeline provenance (`outputs/chronos_provenance.json`), and `outputs/chronos_2_forecasting_result.json` with the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digests, generator), the model identifier, the immutable model revision and licence, and the runtime identity (Python, `torch`, `transformers`, `pandas`, `numpy`, device, dtype). A BYOD SHA-256 is metadata derived from uploaded bytes and should be retained/disclosed under your data-governance rules. No credentials are recorded.

In [ ]:
result.forecast.to_csv("outputs/chronos_forecast.csv", index=False)
mode_c_result.forecast.to_csv("outputs/chronos_multitarget_forecast.csv", index=False)
evaluation.per_series.to_csv("outputs/chronos_evaluation_per_series.csv", index=False)

provenance = {
    **result.provenance,
    "tutorial": {
        "notebook_profile": "TASK-INFERENCE",
        "notebook_spec": NOTEBOOK_SOURCE["notebook_spec"],
        "notebook_source": NOTEBOOK_SOURCE,
        "input_source": input_source,
        "input_sha256": input_sha256,
        "primary_capability_checks": ["univariate", "multi-target"],
    },
}
with open("outputs/chronos_provenance.json", "w", encoding="utf-8") as handle:
    json.dump(provenance, handle, indent=2, default=str)

metrics = {
    "evidence_scope": "tutorial/sanity; not benchmark or production-fitness evidence",
    "estimation_procedure": "single chronological tail holdout",
    "chronos2": evaluation.aggregate,
    "last_value": last_value_evaluation.aggregate,
    "seasonal_naive_24": seasonal_metrics,
    "quantiles": evaluation.quantiles.to_dict("records"),
}
with open("outputs/chronos_evaluation.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2, default=str)

payload_out = {
    "forecast_rows": result.forecast.to_dict("records"),
    "evaluation_report": report,
    "input_manifest": input_manifest,
    "sample": {"kind": sample_kind, "source": input_source, "sha256": input_sha256, "prediction_length": PREDICTION_LENGTH},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_license": MODEL_LICENSE,
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "pandas": pandas.__version__,
        "numpy": numpy.__version__,
        "device": pipe.device,
        "dtype": pipe.dtype,
    },
}
with open("outputs/chronos_2_forecasting_result.json", "w", encoding="utf-8") as handle:
    json.dump(payload_out, handle, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir("outputs")))

## Optional: Mode D known-future covariates

Enable `RUN_COVARIATE_DEMO` or set `DIMER_RUN_COVARIATE_DEMO=1`. The deterministic two-series demand table (the repository's `chronos_covariates_history.csv` / `chronos_covariates_future.csv` formulas, built in memory) carries `temperature` and `holiday` as covariates; the future table contains only those two columns for the 24-step horizon, never future `demand`, so a target column there would be refused as leakage. The provenance names which covariates were read as known-future.

In [ ]:
RUN_COVARIATE_DEMO = False  # @param {type:"boolean"}
if os.environ.get("DIMER_RUN_COVARIATE_DEMO") == "1":
    RUN_COVARIATE_DEMO = True

if RUN_COVARIATE_DEMO:
    n, horizon = 96, 24
    step = np.arange(n, dtype=float)
    future_step = np.arange(n, n + horizon, dtype=float)
    timestamps = pd.date_range("2026-01-01", periods=n, freq="h")
    future_timestamps = pd.date_range(timestamps[-1] + pd.Timedelta(hours=1), periods=horizon, freq="h")
    history_parts, future_parts = [], []
    for series_id, base, phase in (("A", 80.0, 0.0), ("B", 110.0, 4.0)):
        temperature = 27.0 + 4.0 * np.sin(2.0 * np.pi * (step + phase) / 24.0)
        holiday = (pd.Series(timestamps).dt.dayofweek >= 5).astype(int).to_numpy()
        demand = base + 1.8 * temperature + 10.0 * holiday + 0.08 * step + 3.0 * np.sin(2.0 * np.pi * step / 12.0)
        history_parts.append(pd.DataFrame({"series_id": series_id, "timestamp": timestamps, "demand": demand, "temperature": temperature, "holiday": holiday}))
        future_temperature = 27.0 + 4.0 * np.sin(2.0 * np.pi * (future_step + phase) / 24.0)
        future_holiday = (pd.Series(future_timestamps).dt.dayofweek >= 5).astype(int).to_numpy()
        future_parts.append(pd.DataFrame({"series_id": series_id, "timestamp": future_timestamps, "temperature": future_temperature, "holiday": future_holiday}))
    cov_history = pd.concat(history_parts, ignore_index=True)
    cov_future = pd.concat(future_parts, ignore_index=True)
    assert "demand" not in cov_future.columns
    cov_config = ForecastConfig(target="demand", prediction_length=horizon, quantile_levels=[0.1, 0.5, 0.9])
    cov_manifest = validate_inputs(cov_history, cov_config, pipe, future_df=cov_future)
    cov_result = forecast(cov_history, cov_config, pipe, cov_future)
    known_future = cov_result.inference["known_future_covariate_names"]
    print("known-future covariates:", known_future, "| manifest:", cov_manifest["known_future_covariate_names"])
    cov_result.forecast.to_csv("outputs/chronos_covariate_forecast.csv", index=False)
else:
    print("Mode D demo skipped.")

## Interpretation and limits

A successful default run **proves** that the recorded repository revision's package, carried in this notebook, can install the pinned runtime, acquire and digest-verify the pinned model, preserve the chronological evaluation boundary, validate the demonstrated input, execute the public API for univariate and primary multi-target forecasting, compute the documented tutorial metrics/baselines, and export machine-readable forecasts/provenance in the tested runtime. Successful execution proves that the recorded repository revision's package, carried in this notebook, can do exactly that — without the repository being reachable — and no more.

It **does not prove** accuracy, calibration, robustness, fairness, safety, or production readiness for your domain. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain. The synthetic sample is intentionally simple; its metrics are not benchmark evidence and the evaluation report says `sample-sanity` for that reason. `prediction` is the median of an uncalibrated quantile forecast; the requested quantiles are not calibrated intervals and the pipeline ships no threshold. BYOD requires representative multi-origin backtesting, leakage controls, domain baselines, calibration checks, and operational review.

Current limits include fixed-width regular frequencies; monthly/quarterly/yearly/business-day, irregular, and gappy calendars are outside this path, and the pipeline provides no classification, anomaly-detection, imputation, embedding, or training capability.

**Next experiments:** enable `USE_BYOD` with a multi-series hourly CSV of your own and compare the report's `evaluate_forecast` MAE against the `last_value_baseline` and `seasonal_naive_baseline` entries; raise `PREDICTION_LENGTH` toward the native 1,024-step horizon and watch `effective_prediction_length`; enable `RUN_COVARIATE_DEMO` and compare the Mode D forecast with and without the future covariate table.

## References

- Repository README: https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline/blob/main/MODEL_CARD.md
- Sample dataset card: https://github.com/kurtvalcorza/chronos-2-forecasting-pipeline/blob/main/examples/sample-data/DATASET_CARD.md
- Upstream model: https://huggingface.co/amazon/chronos-2
- Upstream library: https://github.com/amazon-science/chronos-forecasting
- Chronos-2: From Univariate to Universal Forecasting: https://arxiv.org/abs/2510.15821